# CitrusHVT · End-to-End Training Notebook

This Colab-ready notebook contains the entire CitrusHVT training/evaluation pipeline. Upload the project folders to Google Drive (same layout as `/Users/letaotao/Desktop/期中答辩`) and follow the steps below to train directly inside this notebook without calling external scripts.

## 1. Mount Google Drive

## 2. Configure paths (edit if your Drive layout differs)

In [1]:
from pathlib import Path
import os

# Change this if you stored the project somewhere else in Drive.
BASE_DIR = Path('/kaggle/input/hvtcitrus')


DATA_ROOT = Path('/kaggle/input/citrus-split/Citrus_Split')
TRAIN_JSON = BASE_DIR / '结构化数据.json'
OOD_ROOT = BASE_DIR / 'OOD' / 'PlantVillage_nonCitrus'
OUTPUT_DIR = Path('/kaggle/working')

for name, path in {
    'DATA_ROOT': DATA_ROOT,
    'TRAIN_JSON': TRAIN_JSON,
    'OOD_ROOT': OOD_ROOT,
}.items():
    status = 'OK' if path.exists() else 'MISSING'
    print(f"{name}: {path} -> {status}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be saved to: {OUTPUT_DIR}")

DATA_ROOT: /kaggle/input/citrus-split/Citrus_Split -> OK
TRAIN_JSON: /kaggle/input/hvtcitrus/结构化数据.json -> OK
OOD_ROOT: /kaggle/input/hvtcitrus/OOD/PlantVillage_nonCitrus -> OK
Outputs will be saved to: /kaggle/working


## 3. Install dependencies

In [2]:
%pip uninstall -y torch torchvision
%pip install --no-cache-dir --upgrade \
  numpy==1.26.4 scipy==1.11.4 \
  torch==2.1.2 torchvision==0.16.2 \
  timm==0.9.12 scikit-learn==1.3.2 \
  pyyaml==6.0.1 albumentations==1.4.4 \
  opencv-python==4.10.0.84 matplotlib==3.7.5

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 110.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 291.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.2/670.2 MB 205.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 218.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 367.2 MB/s eta 0:00:00
   ━━━━━━

In [3]:
import torch
print(torch.__version__, torch.__file__)

2.1.2+cu121 /usr/local/lib/python3.11/dist-packages/torch/__init__.py


## 4. Core imports

In [4]:
import copy
import json
import math
import os
import platform
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score
from torch import nn
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision import io as tvio
from torchvision.transforms import functional as TF
from sklearn.model_selection import StratifiedShuffleSplit
try:
    import timm
except ImportError as exc:
    raise ImportError('timm is required. Install it via `pip install timm`.') from exc

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

## 5. Utility helpers

In [5]:
class AverageMeter:
    def __init__(self) -> None:
        self.reset()

    def reset(self) -> None:
        self.sum = 0.0
        self.count = 0

    def update(self, value: float, n: int = 1) -> None:
        self.sum += value * n
        self.count += n

    @property
    def avg(self) -> float:
        return self.sum / max(1, self.count)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
def _unwrap_model(model: nn.Module) -> nn.Module:
    """获取被 DataParallel 包装的原始模型"""
    return model.module if isinstance(model, (nn.DataParallel, nn.parallel.DistributedDataParallel)) else model


def save_checkpoint(model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    # 保存时去掉 DataParallel 的 module. 前缀，方便单卡加载
    raw_model = _unwrap_model(model)
    state = {
        'epoch': epoch,
        'model': raw_model.state_dict(),
        'optimizer': optimizer.state_dict(),
    }
    torch.save(state, path)

## 6. Data pipeline

In [6]:
def _load_metadata(path: Path) -> Dict[str, Dict]:
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    mapping: Dict[str, Dict] = {}
    if isinstance(data, dict):
        for key, val in data.items():
            mapping[key] = val if isinstance(val, dict) else {'value': val}
    elif isinstance(data, list):
        for entry in data:
            if not isinstance(entry, dict):
                continue
            file_key = entry.get('filename') or entry.get('path') or entry.get('名称')
            if file_key:
                mapping[str(file_key)] = entry
    return mapping


def build_transforms(image_size: int = 224, is_train: bool = True) -> transforms.Compose:
    if is_train:
        return transforms.Compose([
            transforms.Resize(int(image_size * 1.1)),
            transforms.RandomResizedCrop(image_size, scale=(0.7, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.Resize(int(image_size * 1.1)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def _gather_leaf_dirs(root: Path) -> List[Tuple[str, Path, str]]:
    """收集叶子目录（类别目录），支持单层和嵌套结构"""
    entries: List[Tuple[str, Path, str]] = []
    for coarse_dir in sorted([p for p in root.iterdir() if p.is_dir()]):
        subdirs = sorted([p for p in coarse_dir.iterdir() if p.is_dir()])
        if subdirs:
            # 嵌套结构：coarse_dir/leaf_dir
            for leaf in subdirs:
                relative = leaf.relative_to(root)
                class_label = relative.as_posix()
                entries.append((coarse_dir.name, leaf, class_label))
        else:
            # 单层结构：coarse_dir 本身就是类别目录
            relative = coarse_dir.relative_to(root)
            entries.append((coarse_dir.name, coarse_dir, relative.as_posix()))
    return entries


def _build_nested_index(root: Path, class_to_idx_override: Optional[Dict[str, int]] = None) -> Tuple[List[Tuple[Path, int, str, str]], List[str]]:
    """构建嵌套索引，支持单层和嵌套目录结构"""
    samples: List[Tuple[Path, int, str, str]] = []
    classes: List[str] = []
    
    if class_to_idx_override:
        # 如果提供了类别映射（用于验证集），使用它
        class_to_idx = class_to_idx_override
        classes = [k for k, v in sorted(class_to_idx.items(), key=lambda item: item[1])]
    else:
        # 否则创建新的类别映射（用于训练集）
        class_to_idx: Dict[str, int] = {}

    for coarse_name, leaf_dir, class_label in _gather_leaf_dirs(root):
        if class_to_idx_override:
            # 如果是验证集，且遇到训练集没有的类别，跳过
            if class_label not in class_to_idx:
                continue 
            class_idx = class_to_idx[class_label]
        else:
            class_idx = class_to_idx.setdefault(class_label, len(class_to_idx))
            if class_idx == len(classes):
                classes.append(class_label)
        
        for file_path in sorted(leaf_dir.rglob('*')):
            if file_path.is_file() and has_file_allowed_extension(file_path.name, IMG_EXTENSIONS):
                samples.append((file_path, class_idx, coarse_name, class_label))
                
    if not samples:
        print(f"Warning: Found 0 valid images in {root}")
    
    return samples, classes


class CitrusDataset(Dataset):
    def __init__(
        self,
        samples: List[Tuple[Path, int, str, str]],
        classes: List[str],
        indices: Optional[Sequence[int]],
        transform,
        metadata_path: Optional[Path] = None,
        group_field: str = 'group',
        coarse_to_idx: Optional[Dict[str, int]] = None,
        fine_to_coarse: Optional[List[int]] = None,
    ) -> None:
        super().__init__()
        self.samples = samples
        self.indices = list(indices) if indices is not None else list(range(len(samples)))
        self.transform = transform
        self.loader = default_loader
        self.group_field = group_field
        self.metadata = _load_metadata(metadata_path) if metadata_path else {}
        self.classes = classes
        self.coarse_to_idx = coarse_to_idx or {}
        self.fine_to_coarse = fine_to_coarse

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> Dict:
        real_idx = self.indices[idx]
        path, label, coarse_name, class_label = self.samples[real_idx]
        # 修复PIL调色板图像警告：先转换为RGB，再确保是RGB模式
        image = self.loader(str(path))
        if image.mode != 'RGB':
            image = image.convert('RGB')
        if self.transform:
            image = self.transform(image)
        meta_key = Path(path).name
        metadata = self.metadata.get(meta_key, {})
        group = metadata.get(self.group_field) or coarse_name
        # 层次分类：通过 fine_to_coarse 映射获取粗类别标签（兼容平铺和嵌套结构）
        if self.fine_to_coarse is not None:
            coarse_target = self.fine_to_coarse[label]
        else:
            coarse_target = self.coarse_to_idx.get(coarse_name, 0) if self.coarse_to_idx else 0
        return {
            'image': image,
            'target': label,
            'coarse_target': coarse_target,
            'path': str(path),
            'group': group,
            'class_name': class_label,
        }


@dataclass
class DataConfig:
    root: str
    train_json: Optional[str] = None
    val_split: float = 0.15
    batch_size: int = 32
    num_workers: int = 8
    image_size: int = 224


from sklearn.model_selection import StratifiedShuffleSplit

def _split_indices(num_samples: int, labels: List[int], val_split: float, seed: int) -> Tuple[List[int], List[int]]:
    """分层抽样：保证训练集/验证集类别分布一致"""
    stratifier = StratifiedShuffleSplit(
        n_splits=1,
        test_size=val_split,
        random_state=seed
    )
    train_idx, val_idx = next(stratifier.split(range(num_samples), labels))
    return train_idx.tolist(), val_idx.tolist()


# ===== 柑橘病害分类学知识 (Domain Knowledge) =====
# 基于植物病理学的分类体系，将细粒度类别映射到粗粒度大类
# 这是领域专家知识的直接编码，不是通用分组
CITRUS_COARSE_TAXONOMY = {
    # 缺素类 (Nutrient Deficiency) - 共同视觉特征：叶片均匀变色、失绿
    'Deficiency_Nitrogen': '缺素',
    'Deficiency_Iron': '缺素',
    'Deficiency_Zinc': '缺素',
    'Deficiency_Magnesium': '缺素',
    'Deficiency_Manganese': '缺素',
    
    # 病害类 (Disease) - 共同视觉特征：局部病斑、组织坏死
    'Anthracnose': '病害',       # 炭疽病
    'Melanose': '病害',          # 树脂病
    'Citrus_Canker': '病害',     # 溃疡病 ★重庆重点
    'Sooty_Mold': '病害',        # 煤烟病
    'Greasy_Spot': '病害',       # 脂点黄斑病
    'Algal_Leaf_Spot': '病害',   # 青苔病
    'Brown_Spot': '病害',        # 褐斑病
    'HLB': '病害',               # 黄龙病 ★重庆重点
    
    # 虫害类 (Pest) - 共同视觉特征：虫蚀痕迹、虫体附着
    'Red_Spider': '虫害',        # 红蜘蛛 ★重庆重点
    'Leaf_Miner': '虫害',        # 潜叶蛾 ★重庆重点
    'Aphids': '虫害',            # 蚜虫
    'Scale_Insect': '虫害',      # 吹绵蚧
    
    # 健康 (如果有)
    'Healthy': '健康',
}

# 重庆重点关注类别（三峡库区高发病虫害）
CHONGQING_FOCUS_CLASSES = {
    'Citrus_Canker',    # 溃疡病 - 检疫性病害，重庆柑橘第一大病害
    'Anthracnose',      # 炭疽病 - 高湿环境多发，重庆夏季高温高湿
    'Melanose',         # 树脂病 - 老龄果园多发
    'HLB',              # 黄龙病 - 毁灭性病害，近年向重庆蔓延
    'Red_Spider',       # 红蜘蛛 - 重庆柑橘第一大虫害
    'Leaf_Miner',       # 潜叶蛾 - 夏秋嫩梢期严重
    'Aphids',           # 蚜虫 - 春梢期高发
    'Deficiency_Iron',  # 缺铁 - 重庆紫色土偏碱，缺铁常见
    'Deficiency_Zinc',  # 缺锌 - 山地柑橘多发
    'Deficiency_Magnesium',  # 缺镁 - 酸性红壤多发
}


def _infer_coarse_from_classname(class_name: str) -> str:
    """根据类名推断粗类别（兼容目录结构和平铺结构）
    
    策略：
    1. 先查精确映射表 CITRUS_COARSE_TAXONOMY
    2. 再查带路径的类名（如 '缺素/Deficiency_Nitrogen'）
    3. 最后用前缀匹配（Deficiency_* → 缺素）
    """
    # 精确匹配
    if class_name in CITRUS_COARSE_TAXONOMY:
        return CITRUS_COARSE_TAXONOMY[class_name]
    
    # 如果class_name带路径前缀（嵌套目录结构），提取最后一段
    basename = class_name.split('/')[-1]
    if basename in CITRUS_COARSE_TAXONOMY:
        return CITRUS_COARSE_TAXONOMY[basename]
    
    # 前缀推断
    lower = class_name.lower()
    if 'deficiency' in lower or '缺' in lower:
        return '缺素'
    if any(k in lower for k in ['spider', 'miner', 'aphid', 'scale', 'insect', '蛾', '蚜', '螨', '蚧']):
        return '虫害'
    if 'healthy' in lower or '健康' in lower:
        return '健康'
    # 默认归为病害
    return '病害'


def _build_coarse_mapping(samples: List[Tuple[Path, int, str, str]], classes: List[str]) -> Dict:
    """构建层次分类映射（柑橘病害分类学知识编码）
    
    基于植物病理学分类体系，将17个细粒度类别映射到3个粗粒度大类：
    - 缺素类：缺氮、缺铁、缺锌、缺镁、缺锰
    - 病害类：溃疡病、炭疽病、树脂病等
    - 虫害类：红蜘蛛、潜叶蛾、蚜虫等
    
    先尝试从目录结构提取层次关系，如果目录是平铺结构则根据类名推断。
    """
    # 1. 先从样本的coarse_name字段尝试提取
    fine_idx_to_coarse_name: Dict[int, str] = {}
    for _, class_idx, coarse_name, _ in samples:
        if class_idx not in fine_idx_to_coarse_name:
            fine_idx_to_coarse_name[class_idx] = coarse_name
    
    # 2. 检测是否是平铺结构（coarse_name == class_name，即没有真正的层次）
    unique_coarse_from_dir = set(fine_idx_to_coarse_name.values())
    is_flat = len(unique_coarse_from_dir) == len(classes)  # 粗类别数 == 细类别数 → 平铺
    
    if is_flat:
        print(f"  📂 检测到平铺目录结构，使用柑橘病害分类学知识推断层次关系")
        # 用领域知识推断粗类别
        for class_idx in range(len(classes)):
            class_name = classes[class_idx]
            fine_idx_to_coarse_name[class_idx] = _infer_coarse_from_classname(class_name)
    else:
        print(f"  📂 检测到嵌套目录结构，直接使用目录层次")
    
    # 3. 构建粗类别索引
    unique_coarse = sorted(set(fine_idx_to_coarse_name.values()))
    coarse_to_idx = {name: i for i, name in enumerate(unique_coarse)}
    coarse_classes = unique_coarse
    
    # 4. 构建 fine → coarse 映射数组
    fine_to_coarse = [coarse_to_idx[fine_idx_to_coarse_name[i]] for i in range(len(classes))]
    
    # 5. 打印映射关系
    print(f"  🌳 层次分类映射（柑橘病害分类学）:")
    print(f"     粗类别 ({len(coarse_classes)}): {coarse_classes}")
    for coarse_name in coarse_classes:
        fine_names = [classes[i] for i in range(len(classes)) if fine_idx_to_coarse_name.get(i) == coarse_name]
        cq_markers = []
        for fn in fine_names:
            basename = fn.split('/')[-1]
            marker = ' ★' if basename in CHONGQING_FOCUS_CLASSES else ''
            cq_markers.append(f"{fn}{marker}")
        print(f"     {coarse_name} ({len(fine_names)}类): {cq_markers}")
    
    # 6. 统计重庆重点类别覆盖
    all_class_basenames = [c.split('/')[-1] for c in classes]
    cq_covered = [c for c in all_class_basenames if c in CHONGQING_FOCUS_CLASSES]
    print(f"  🏙️ 重庆重点类别覆盖: {len(cq_covered)}/{len(CHONGQING_FOCUS_CLASSES)} ({cq_covered})")
    
    return {
        'coarse_to_idx': coarse_to_idx,
        'coarse_classes': coarse_classes,
        'fine_to_coarse': fine_to_coarse,
        'num_coarse_classes': len(coarse_classes),
    }


def build_dataloaders(cfg: DataConfig, seed: int = 42, pin_memory: bool = True) -> Tuple[DataLoader, DataLoader, List[int], Dict]:
    """返回: (train_loader, val_loader, all_targets, coarse_info)"""
    root = Path(cfg.root)
    train_dir = root / 'train'
    val_dir = root / 'val'
    test_dir = root / 'test'
    
    # 检查是否已经是分好类的结构（Citrus_Split 格式）
    if train_dir.exists() and val_dir.exists():
        print(f"检测到已分好的数据集结构: {root}")
        print(f"  训练集目录: {train_dir}")
        print(f"  验证集目录: {val_dir}")
        
        # 1. 加载训练集，建立类别映射
        train_samples, classes = _build_nested_index(train_dir)
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        print(f"  找到 {len(train_samples)} 个训练样本，{len(classes)} 个类别")

        # 统计训练集类别分布
        train_class_counts = Counter([s[1] for s in train_samples])
        print(f"  训练集类别分布: {dict(sorted(train_class_counts.items()))}")
        
        # 2. 加载验证集，强制使用训练集的类别映射
        val_samples, _ = _build_nested_index(val_dir, class_to_idx_override=class_to_idx)
        print(f"  找到 {len(val_samples)} 个验证样本")

        # 统计验证集类别分布
        val_class_counts = Counter([s[1] for s in val_samples])
        print(f"  验证集类别分布: {dict(sorted(val_class_counts.items()))}")
        
        # 3. 如果存在测试集，将其合并到验证集中
        if test_dir.exists():
            print(f"  检测到测试集目录: {test_dir}")
            test_samples, _ = _build_nested_index(test_dir, class_to_idx_override=class_to_idx)
            print(f"  找到 {len(test_samples)} 个测试样本")
            
            # 合并验证集和测试集
            val_samples = val_samples + test_samples
            print(f"  合并后验证集总样本数: {len(val_samples)}")
            
            # 统计合并后的验证集类别分布
            combined_val_class_counts = Counter([s[1] for s in val_samples])
            print(f"  合并后验证集类别分布: {dict(sorted(combined_val_class_counts.items()))}")
        else:
            print(f"  未检测到测试集目录，仅使用验证集")

        # 构建层次分类映射（柑橘病害分类学知识）
        coarse_info = _build_coarse_mapping(train_samples, classes)
        coarse_to_idx = coarse_info['coarse_to_idx']
        
        # 4. 构造 Dataset（不需要再随机切分）
        train_idx = list(range(len(train_samples)))
        val_idx = list(range(len(val_samples)))
        
        metadata_path = Path(cfg.train_json) if cfg.train_json else None
        train_tf = build_transforms(cfg.image_size, True)
        val_tf = build_transforms(cfg.image_size, False)
        
        train_dataset = CitrusDataset(train_samples, classes, train_idx, train_tf, 
                                      metadata_path=metadata_path, coarse_to_idx=coarse_to_idx,
                                      fine_to_coarse=coarse_info['fine_to_coarse'])
        val_dataset = CitrusDataset(val_samples, classes, val_idx, val_tf, 
                                    metadata_path=metadata_path, coarse_to_idx=coarse_to_idx,
                                    fine_to_coarse=coarse_info['fine_to_coarse'])
        
        # 4. 构造所有样本的标签列表（用于后续分析）
        all_targets = [sample[1] for sample in train_samples] + [sample[1] for sample in val_samples]
        
    else:
        # 老逻辑：从单个根目录自动划分
        print(f"从单个根目录自动划分数据集: {root}")
        samples, classes = _build_nested_index(root)

        # 构建层次分类映射
        coarse_info = _build_coarse_mapping(samples, classes)
        coarse_to_idx = coarse_info['coarse_to_idx']
        
        # 获取所有样本的标签
        labels = [sample[1] for sample in samples]
        
        # 使用分层抽样划分训练集和验证集
        train_idx, val_idx = _split_indices(len(samples), labels, cfg.val_split, seed)
        
        metadata_path = Path(cfg.train_json) if cfg.train_json else None
        train_tf = build_transforms(cfg.image_size, True)
        val_tf = build_transforms(cfg.image_size, False)
        
        train_dataset = CitrusDataset(samples, classes, train_idx, train_tf, 
                                      metadata_path=metadata_path, coarse_to_idx=coarse_to_idx,
                                      fine_to_coarse=coarse_info['fine_to_coarse'])
        val_dataset = CitrusDataset(samples, classes, val_idx, val_tf, 
                                    metadata_path=metadata_path, coarse_to_idx=coarse_to_idx,
                                    fine_to_coarse=coarse_info['fine_to_coarse'])
        
        all_targets = labels

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
    )
    
    return train_loader, val_loader, all_targets, coarse_info

## 7. Wavelet-CNN building blocks

In [7]:
class HaarDWT(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        low = torch.tensor([1.0 / math.sqrt(2.0), 1.0 / math.sqrt(2.0)])
        high = torch.tensor([1.0 / math.sqrt(2.0), -1.0 / math.sqrt(2.0)])
        filters = torch.stack([
            torch.outer(low, low),
            torch.outer(low, high),
            torch.outer(high, low),
            torch.outer(high, high),
        ], dim=0)
        self.register_buffer('filters', filters.reshape(4, 1, 2, 2), persistent=False)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        b, c, h, w = x.shape
        filters = self.filters.to(dtype=x.dtype)
        x_flat = x.reshape(b * c, 1, h, w)
        y = F.conv2d(x_flat, filters, stride=2)
        y = y.reshape(b, c, 4, h // 2, w // 2)
        ll = y[:, :, 0]
        lh = y[:, :, 1]
        hl = y[:, :, 2]
        hh = y[:, :, 3]
        return ll, lh, hl, hh


class WaveConvBlock(nn.Module):
    def __init__(self, channels: int, hidden_channels: int = 32, dropout: float = 0.0) -> None:
        super().__init__()
        self.dwt = HaarDWT()
        self.low_conv = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(hidden_channels),
            nn.GELU(),
        )
        self.freq_conv = nn.Sequential(
            nn.Conv2d(channels * 3, hidden_channels * 2, 1, bias=False),
            nn.BatchNorm2d(hidden_channels * 2),
            nn.GELU(),
        )
        fused_channels = channels + hidden_channels + hidden_channels * 2
        self.mix = nn.Sequential(
            nn.Conv2d(fused_channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        ll, lh, hl, hh = self.dwt(x)
        low = self.low_conv(ll)
        freq = torch.cat([lh, hl, hh], dim=1)
        freq = self.freq_conv(freq)
        low = F.interpolate(low, size=x.shape[-2:], mode='bilinear', align_corners=False)
        freq = F.interpolate(freq, size=x.shape[-2:], mode='bilinear', align_corners=False)
        fused = torch.cat([x, low, freq], dim=1)
        out = self.mix(fused)
        return out + identity


class WaveletStem(nn.Module):
    def __init__(self, in_channels: int = 3, hidden_channels: int = 48, num_blocks: int = 2, dropout: float = 0.0) -> None:
        super().__init__()
        self.blocks = nn.ModuleList([
            WaveConvBlock(in_channels, hidden_channels, dropout=dropout) for _ in range(num_blocks)
        ])
        self.out_norm = nn.BatchNorm2d(in_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for blk in self.blocks:
            x = blk(x)
        return self.out_norm(x)


class LocalFeatureExtractor(nn.Module):
    def __init__(
        self,
        backbone_name: str = 'convnext_tiny.fb_in22k_ft_in1k',
        img_size: int = 224,
        pretrained: bool = True,
        drop_path: float = 0.1,
        use_wavelet: bool = True,
        use_multiscale: bool = True,
        local_dim: int = 512,
    ) -> None:
        super().__init__()
        if timm is None:
            raise ImportError('timm is required for LocalFeatureExtractor')
        self.use_wavelet = use_wavelet
        self.use_multiscale = use_multiscale
        self.wavelet_stem = WaveletStem(num_blocks=2, dropout=0.05) if use_wavelet else nn.Identity()
        backbone_kwargs = dict(
            pretrained=pretrained,
            features_only=True,
            out_indices=(1, 2, 3),
            drop_path_rate=drop_path,
        )
        if 'vit' in backbone_name or 'swin' in backbone_name:
            backbone_kwargs['img_size'] = img_size
        self.backbone = timm.create_model(backbone_name, **backbone_kwargs)
        feature_channels: Sequence[int] = self.backbone.feature_info.channels()
        if len(feature_channels) < 3:
            raise ValueError('Backbone must expose at least three feature levels for C2/C3/C4 fusion.')
        fusion_in = sum(feature_channels[-3:]) if use_multiscale else feature_channels[-1]
        self.fusion = nn.Sequential(
            nn.Conv2d(fusion_in, local_dim, 1, bias=False),
            nn.BatchNorm2d(local_dim),
            nn.GELU(),
        )
        self.out_dim = local_dim

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        x = self.wavelet_stem(x)
        features = self.backbone(x)[-3:]
        if self.use_multiscale:
            target_hw = features[0].shape[-2:]
            processed = [F.interpolate(feat, size=target_hw, mode='bilinear', align_corners=False) for feat in features]
            fused = torch.cat(processed, dim=1)
        else:
            fused = features[-1]
        fused = self.fusion(fused)
        pooled = fused.mean(dim=(-2, -1))
        return pooled, features

## 8. Cross-branch fusion + classifier

In [8]:
class CrossBranchGate(nn.Module):
    def __init__(self, local_dim: int, global_dim: int, d_model: int = 384) -> None:
        super().__init__()
        self.local_proj = nn.Linear(local_dim, d_model)
        self.global_proj = nn.Linear(global_dim, d_model)
        self.attn_mlp = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, 2),
        )
        self.cross_mlp = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.out_norm = nn.LayerNorm(d_model)

    def forward(self, f_local: torch.Tensor, f_global: torch.Tensor) -> torch.Tensor:
        l = self.local_proj(f_local)
        g = self.global_proj(f_global)
        weights = self.attn_mlp(torch.cat([l, g], dim=-1))
        weights = torch.softmax(weights, dim=-1)
        fused = weights[:, :1] * l + weights[:, 1:2] * g
        bridge = self.cross_mlp(torch.abs(l - g))
        out = fused + bridge
        return self.out_norm(out)


class GlobalFeatureExtractor(nn.Module):
    def __init__(self, vit_name: str = 'vit_base_patch16_224', img_size: int = 224, pretrained: bool = True) -> None:
        super().__init__()
        if timm is None:
            raise ImportError('timm is required for GlobalFeatureExtractor')
        self.encoder = timm.create_model(vit_name, pretrained=pretrained, img_size=img_size)
        if hasattr(self.encoder, 'reset_classifier'):
            self.encoder.reset_classifier(0)
        self.out_dim = getattr(self.encoder, 'num_features', 768)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.encoder.forward_features(x)
        if isinstance(feats, (tuple, list)):
            feats = feats[0]
        if feats.dim() == 3:
            cls_token = feats[:, 0]
        else:
            cls_token = feats.mean(dim=(-2, -1))
        return cls_token


@dataclass
class ModelConfig:
    num_classes: int = 21
    d_model: int = 384  # 降低以匹配 ViT-Tiny
    local_dim: int = 256  # 降低以加速训练
    global_dim: int = 192  # 匹配 ViT-Tiny 的输出维度
    vit_name: str = 'vit_tiny_patch16_224'  # 切换到 ViT-Tiny
    baseline: str = 'full'
    img_size: int = 224
    drop_path: float = 0.1
    # 层次分类配置（柑橘病害分类学知识编码）
    use_hierarchical: bool = False
    num_coarse_classes: int = 3  # 缺素、病害、虫害
    fine_to_coarse: Optional[List[int]] = None  # fine_class_idx → coarse_class_idx


class CitrusHVT(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        baseline = config.baseline.lower()
        self.use_local = baseline not in {'global_only'}
        self.use_global = baseline not in {'cnn_only', 'wavecnn_only'}
        if not self.use_local and not self.use_global:
            raise ValueError('At least one branch must be enabled.')

        use_wavelet = baseline not in {'cnn_only', 'no_dwt'}
        use_multiscale = baseline not in {'no_multiscale'}
        local_dim = config.local_dim
        if self.use_local:
            self.local_branch = LocalFeatureExtractor(
                img_size=config.img_size,
                drop_path=config.drop_path,
                use_wavelet=use_wavelet,
                use_multiscale=use_multiscale,
                local_dim=local_dim,
            )
        else:
            self.local_branch = None

        if self.use_global:
            self.global_branch = GlobalFeatureExtractor(
                vit_name=config.vit_name,
                img_size=config.img_size,
            )
        else:
            self.global_branch = None

        gate_disabled = baseline in {'concat', 'no_gate'} or not (self.use_local and self.use_global)
        self.use_gate = not gate_disabled and baseline not in {'concat'}

        if self.use_local and not self.use_global:
            projector_in = local_dim
        elif self.use_global and not self.use_local:
            projector_in = self.global_branch.out_dim
        elif self.use_gate:
            self.gate = CrossBranchGate(local_dim, self.global_branch.out_dim, d_model=config.d_model)
            projector_in = config.d_model
        else:
            projector_in = local_dim + self.global_branch.out_dim

        self.projector = nn.Sequential(
            nn.LayerNorm(projector_in),
            nn.Linear(projector_in, config.d_model),
            nn.GELU(),
        )
        self.classifier = nn.Linear(config.d_model, config.num_classes)

        # ===== 层次分类头（柑橘病害分类学知识编码）=====
        # 模拟植物病理专家的诊断流程：先判大类(缺素/病害/虫害)，再判细类
        self.use_hierarchical = config.use_hierarchical
        if self.use_hierarchical:
            self.coarse_classifier = nn.Sequential(
                nn.Linear(config.d_model, config.d_model // 2),
                nn.GELU(),
                nn.Dropout(0.1),
                nn.Linear(config.d_model // 2, config.num_coarse_classes),
            )
            # 注册 fine→coarse 映射（不可训练）
            if config.fine_to_coarse is not None:
                self.register_buffer(
                    'fine_to_coarse',
                    torch.tensor(config.fine_to_coarse, dtype=torch.long)
                )
            else:
                self.fine_to_coarse = None


    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        features: Dict[str, torch.Tensor] = {}
        if self.use_local:
            f_local, multi_scale = self.local_branch(x)
            features['f_local'] = f_local
            features['c_feats'] = multi_scale
        if self.use_global:
            f_global = self.global_branch(x)
            features['f_global'] = f_global

        if self.use_local and not self.use_global:
            fused = f_local
        elif self.use_global and not self.use_local:
            fused = f_global
        elif self.use_gate:
            fused = self.gate(features['f_local'], features['f_global'])
        else:
            fused = torch.cat([features['f_local'], features['f_global']], dim=-1)

        proj = self.projector(fused)
        logits = self.classifier(proj)
        features['embedding'] = proj
        features['logits'] = logits

        # 层次分类：粗类别预测 + 层次一致性
        if self.use_hierarchical:
            coarse_logits = self.coarse_classifier(proj)
            features['coarse_logits'] = coarse_logits
            # 计算层次一致性：细分类的概率在粗类别上的聚合 vs 粗分类头的直接预测
            if self.fine_to_coarse is not None:
                fine_probs = F.softmax(logits, dim=-1)  # [B, num_fine]
                # 聚合：将细类别概率按粗类别求和
                num_coarse = coarse_logits.size(-1)
                implied_coarse = torch.zeros(fine_probs.size(0), num_coarse, device=logits.device)
                implied_coarse.scatter_add_(1, self.fine_to_coarse.unsqueeze(0).expand_as(fine_probs), fine_probs)
                features['implied_coarse_probs'] = implied_coarse
        
        return features

## 9. Losses and metrics

In [9]:
def compute_class_weights(samples_per_class: List[int], beta: float) -> torch.Tensor:
    effective_num = [1.0 - beta ** n for n in samples_per_class]
    weights = [(1.0 - beta) / (en if en > 0 else 1.0) for en in effective_num]
    norm = sum(weights)
    if norm > 0:
        weights = [w * len(samples_per_class) / norm for w in weights]
    return torch.tensor(weights, dtype=torch.float32)


class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, samples_per_class: List[int], beta: float = 0.99, gamma: float = 2.0, label_smoothing: float = 0.0) -> None:
        super().__init__()
        if len(samples_per_class) == 0:
            raise ValueError('samples_per_class cannot be empty')
        self.class_weights = compute_class_weights(samples_per_class, beta)
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.num_classes = len(samples_per_class)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        weights = self.class_weights.to(logits.device)
        probs = F.softmax(logits, dim=-1)
        log_probs = torch.log(probs.clamp_min(1e-7))
        target_one_hot = F.one_hot(targets, self.num_classes).float()
        if self.label_smoothing > 0:
            target_one_hot = target_one_hot * (1 - self.label_smoothing) + self.label_smoothing / self.num_classes
        ce = -(target_one_hot * log_probs).sum(dim=-1)
        pt = (probs * target_one_hot).sum(dim=-1)
        focal_factor = (1 - pt).pow(self.gamma)
        sample_weights = (weights * target_one_hot).sum(dim=-1)
        return ce * focal_factor * sample_weights


class CVarAggregator(nn.Module):
    def __init__(self, alpha: float = 0.2) -> None:
        super().__init__()
        self.alpha = alpha

    def forward(self, per_sample_loss: torch.Tensor) -> torch.Tensor:
        if self.alpha <= 0:
            return per_sample_loss.mean()
        k = max(1, int(math.ceil(self.alpha * per_sample_loss.size(0))))
        topk, _ = torch.topk(per_sample_loss, k=k, dim=0)
        return topk.mean()


class HierarchicalLoss(nn.Module):
    """层次分类损失函数（柑橘病害领域知识驱动）
    
    编码植物病理学的分层诊断逻辑：
    1. L_fine: 细粒度分类损失（17类具体病害）
    2. L_coarse: 粗粒度分类损失（缺素/病害/虫害3大类）  
    3. L_consist: 层次一致性损失（细分类概率聚合 ≈ 粗分类概率）
    
    Total = L_fine + λ_coarse * L_coarse + λ_consist * L_consist
    
    领域意义：
    - L_coarse 强迫模型先学会区分大类（缺素 vs 病害 vs 虫害），防止跨类混淆
    - L_consist 确保模型的细粒度判断与粗粒度判断语义一致
    - 这是柑橘病害分类体系的直接编码，不是通用技巧
    """
    def __init__(
        self,
        fine_criterion,  # 细分类损失（可以是CB-Focal或CE）
        num_coarse_classes: int = 3,
        lambda_coarse: float = 0.3,
        lambda_consist: float = 0.1,
    ) -> None:
        super().__init__()
        self.fine_criterion = fine_criterion
        self.coarse_criterion = nn.CrossEntropyLoss()
        self.lambda_coarse = lambda_coarse
        self.lambda_consist = lambda_consist
        self.num_coarse_classes = num_coarse_classes
    
    def forward(
        self,
        fine_logits: torch.Tensor,
        coarse_logits: torch.Tensor,
        fine_targets: torch.Tensor,
        coarse_targets: torch.Tensor,
        implied_coarse_probs: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """返回各项损失的字典，方便监控"""
        # 1. 细分类损失
        if isinstance(self.fine_criterion, nn.CrossEntropyLoss):
            fine_loss = self.fine_criterion(fine_logits, fine_targets)
        else:
            per_sample = self.fine_criterion(fine_logits, fine_targets)
            fine_loss = per_sample  # 返回per-sample，后续由CVaR处理
        
        # 2. 粗分类损失（辅助任务：缺素/病害/虫害三分类）
        coarse_loss = self.coarse_criterion(coarse_logits, coarse_targets)
        
        # 3. 层次一致性损失（可选，确保细分类和粗分类语义一致）
        consist_loss = torch.tensor(0.0, device=fine_logits.device)
        if implied_coarse_probs is not None and self.lambda_consist > 0:
            coarse_probs = F.softmax(coarse_logits.detach(), dim=-1)  # detach粗分类梯度
            # KL散度：implied_coarse_probs 应该接近 coarse_probs
            consist_loss = F.kl_div(
                (implied_coarse_probs + 1e-8).log(),
                coarse_probs,
                reduction='batchmean',
            )
        
        return {
            'fine_loss': fine_loss,
            'coarse_loss': coarse_loss,
            'consist_loss': consist_loss,
            'lambda_coarse': self.lambda_coarse,
            'lambda_consist': self.lambda_consist,
        }


def accuracy(preds: torch.Tensor, targets: torch.Tensor) -> float:
    return (preds == targets).float().mean().item()


def macro_f1(preds: torch.Tensor, targets: torch.Tensor, num_classes: int) -> float:
    preds_np = preds.cpu().numpy()
    targets_np = targets.cpu().numpy()
    return f1_score(targets_np, preds_np, average='macro', labels=list(range(num_classes)), zero_division=0)


class GroupMeter:
    def __init__(self) -> None:
        self.storage: Dict[str, Dict[str, List[int]]] = defaultdict(lambda: {'preds': [], 'targets': []})

    def update(self, group_names: List[str], preds: torch.Tensor, targets: torch.Tensor) -> None:
        preds_list = preds.cpu().tolist()
        targets_list = targets.cpu().tolist()
        for g, p, t in zip(group_names, preds_list, targets_list):
            buf = self.storage[g]
            buf['preds'].append(p)
            buf['targets'].append(t)

    def compute(self, num_classes: int) -> Tuple[Dict[str, Dict[str, float]], str]:
        scores: Dict[str, Dict[str, float]] = {}
        worst_group = None
        worst_f1 = 1.0
        for group, buf in self.storage.items():
            preds = torch.tensor(buf['preds'])
            targets = torch.tensor(buf['targets'])
            acc = accuracy(preds, targets)
            f1 = macro_f1(preds, targets, num_classes)
            scores[group] = {'acc': acc, 'macro_f1': f1, 'count': len(buf['preds'])}
            if f1 < worst_f1:
                worst_f1 = f1
                worst_group = group
        return scores, worst_group or 'N/A'

## 10. Evaluation helpers

In [10]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def _denormalize(x: torch.Tensor) -> torch.Tensor:
    return x * IMAGENET_STD.to(x.device) + IMAGENET_MEAN.to(x.device)


def _normalize(x: torch.Tensor) -> torch.Tensor:
    return (x - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)


def apply_jpeg(images: torch.Tensor, quality: int) -> torch.Tensor:
    imgs = (_denormalize(images).clamp(0, 1) * 255).to(torch.uint8).cpu()
    outputs = []
    for img in imgs:
        buf = tvio.encode_jpeg(img, quality=quality)
        rec = tvio.decode_jpeg(buf).float() / 255.0
        outputs.append(rec)
    stacked = torch.stack(outputs, dim=0).to(images.device)
    return _normalize(stacked)


def apply_gaussian_blur(images: torch.Tensor, sigma: float) -> torch.Tensor:
    blurred = TF.gaussian_blur(_denormalize(images), kernel_size=5, sigma=sigma)
    return _normalize(blurred)


def apply_gaussian_noise(images: torch.Tensor, std: float) -> torch.Tensor:
    noisy = _denormalize(images) + torch.randn_like(images) * std
    return _normalize(noisy.clamp(0, 1))


def apply_color_jitter(images: torch.Tensor, level: int) -> torch.Tensor:
    factor = 0.15 * level
    jittered = TF.adjust_brightness(_denormalize(images), 1 + factor)
    jittered = TF.adjust_contrast(jittered, 1 + factor)
    jittered = TF.adjust_saturation(jittered, 1 + factor)
    jittered = TF.adjust_hue(jittered, min(0.1 * level, 0.4))
    return _normalize(jittered.clamp(0, 1))


def evaluate_model(model: nn.Module, dataloader: DataLoader, device: str, num_classes: int, criterion=None, cvar=None, compute_val_loss: bool = True) -> Dict[str, float]:
    """评估模型，返回准确率、F1和验证损失"""
    model.eval()
    preds_all: List[torch.Tensor] = []
    targets_all: List[torch.Tensor] = []
    coarse_preds_all: List[torch.Tensor] = []
    coarse_targets_all: List[torch.Tensor] = []
    group_meter = GroupMeter()
    val_losses = []
    raw_model = _unwrap_model(model)
    use_hierarchical = hasattr(raw_model, 'use_hierarchical') and raw_model.use_hierarchical
    
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device, non_blocking=True)
            targets = batch['target'].to(device, non_blocking=True)
            model_outputs = model(images)
            outputs = model_outputs['logits']
            preds = outputs.argmax(dim=-1)
            preds_all.append(preds.cpu())
            targets_all.append(targets.cpu())
            group_meter.update(batch['group'], preds, targets)

            # 层次分类：收集粗分类预测
            if use_hierarchical and 'coarse_logits' in model_outputs:
                coarse_preds = model_outputs['coarse_logits'].argmax(dim=-1)
                coarse_preds_all.append(coarse_preds.cpu())
                coarse_targets_all.append(batch['coarse_target'])
            
            
            # 计算验证损失（如果提供了criterion）
            if criterion is not None and compute_val_loss:
                if isinstance(criterion, HierarchicalLoss):
                    coarse_targets = batch['coarse_target'].to(device, non_blocking=True)
                    loss_dict = criterion(
                        fine_logits=outputs,
                        coarse_logits=model_outputs.get('coarse_logits', outputs),
                        fine_targets=targets,
                        coarse_targets=coarse_targets,
                        implied_coarse_probs=model_outputs.get('implied_coarse_probs'),
                    )
                    fine_loss = loss_dict['fine_loss']
                    if fine_loss.dim() > 0:
                        fine_loss = fine_loss.mean()
                    batch_loss = (fine_loss 
                                  + loss_dict['lambda_coarse'] * loss_dict['coarse_loss']
                                  + loss_dict['lambda_consist'] * loss_dict['consist_loss'])
                    val_losses.append(batch_loss.item())
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    batch_loss = criterion(outputs, targets)
                    val_losses.append(batch_loss.item())
                elif cvar is not None:
                    per_sample_loss = criterion(outputs, targets)
                    if cvar.alpha <= 0:
                        batch_loss = per_sample_loss.mean()
                    else:
                        batch_loss = cvar(per_sample_loss)
                    val_losses.append(batch_loss.item())
                else:
                    per_sample_loss = criterion(outputs, targets)
                    batch_loss = per_sample_loss.mean()
                    val_losses.append(batch_loss.item())
                    
    preds_tensor = torch.cat(preds_all)
    targets_tensor = torch.cat(targets_all)
    preds_np = preds_tensor.cpu().numpy()
    targets_np = targets_tensor.cpu().numpy()
    labels_list = list(range(num_classes))
    macro_f1_val = f1_score(targets_np, preds_np, average='macro', labels=labels_list, zero_division=0)
    f1_per_class = f1_score(targets_np, preds_np, average=None, labels=labels_list, zero_division=0)
    min_class_f1 = float(np.min(f1_per_class)) if f1_per_class.size else 0.0
    f1_std = float(np.std(f1_per_class)) if f1_per_class.size else 0.0
    scores = {
        'acc': accuracy(preds_tensor, targets_tensor),
        'macro_f1': float(macro_f1_val),
        'min_class_f1': min_class_f1,
        'f1_std': f1_std,
    }
    # 层次分类：计算粗分类准确率
    if coarse_preds_all:
        coarse_preds_tensor = torch.cat(coarse_preds_all)
        coarse_targets_tensor = torch.cat(coarse_targets_all)
        scores['coarse_acc'] = accuracy(coarse_preds_tensor, coarse_targets_tensor)
    
    # 添加验证损失
    if val_losses:
        scores['val_loss'] = sum(val_losses) / len(val_losses)
    group_stats, worst_group = group_meter.compute(num_classes)
    scores['worst_group_f1'] = group_stats.get(worst_group, {}).get('macro_f1', scores['macro_f1'])
    return scores


def evaluate_degradations(model: nn.Module, dataloader: DataLoader, device: str, num_classes: int, cfg: Dict) -> Dict[str, Dict[str, float]]:
    degradations = {}
    for quality in cfg.get('jpeg_quality', []):
        degradations[f'jpeg_{quality}'] = lambda x, q=quality: apply_jpeg(x, q)
    for sigma in cfg.get('gaussian_blur_sigma', []):
        degradations[f'blur_{sigma}'] = lambda x, s=sigma: apply_gaussian_blur(x, s)
    for noise in cfg.get('gaussian_noise_std', []):
        degradations[f'noise_{noise}'] = lambda x, n=noise: apply_gaussian_noise(x, n)
    for level in cfg.get('color_jitter_levels', []):
        degradations[f'color_{level}'] = lambda x, l=level: apply_color_jitter(x, l)

    metrics = {}
    model.eval()
    for name, fn in degradations.items():
        preds_all: List[torch.Tensor] = []
        targets_all: List[torch.Tensor] = []
        with torch.no_grad():
            for batch in dataloader:
                images = fn(batch['image'].to(device))
                targets = batch['target'].to(device)
                logits = model(images)['logits']
                preds = logits.argmax(dim=-1)
                preds_all.append(preds.cpu())
                targets_all.append(targets.cpu())
        preds_tensor = torch.cat(preds_all)
        targets_tensor = torch.cat(targets_all)
        metrics[name] = {
            'acc': accuracy(preds_tensor, targets_tensor),
            'macro_f1': macro_f1(preds_tensor, targets_tensor, _unwrap_model(model).config.num_classes),
        }
    return metrics


def compute_energy(logits: torch.Tensor) -> torch.Tensor:
    return -torch.logsumexp(logits, dim=-1)


def evaluate_ood(model: nn.Module, id_loader: DataLoader, ood_loader: DataLoader, device: str) -> Dict[str, float]:
    model.eval()
    energies: List[float] = []
    labels: List[int] = []
    with torch.no_grad():
        for batch in id_loader:
            logits = model(batch['image'].to(device))['logits']
            energies.extend(compute_energy(logits).cpu().tolist())
            labels.extend([0] * logits.size(0))
        for batch in ood_loader:
            logits = model(batch['image'].to(device))['logits']
            energies.extend(compute_energy(logits).cpu().tolist())
            labels.extend([1] * logits.size(0))
    scores = torch.tensor(energies)
    targets = torch.tensor(labels)
    auroc = roc_auc_score(targets.numpy(), scores.numpy())
    return {
        'auroc': float(auroc),
        'energy_mean_id': float(scores[targets == 0].mean()),
        'energy_mean_ood': float(scores[targets == 1].mean()),
    }


def tent_step(model: nn.Module, images: torch.Tensor, optimizer: torch.optim.Optimizer) -> torch.Tensor:
    model.train()
    outputs = model(images)
    logits = outputs['logits']
    probs = F.softmax(logits, dim=-1)
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1).mean()
    optimizer.zero_grad()
    entropy.backward()
    optimizer.step()
    return logits.detach()

## 11. Training loop

In [11]:
import tqdm # Import tqdm for progress bar

def build_optimizer(model: nn.Module, lr: float, weight_decay: float) -> torch.optim.Optimizer:
    return AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


def train_one_epoch(
    model: CitrusHVT,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion, 
    cvar: CVarAggregator,
    device: str,
    scaler: Optional[GradScaler],
    use_amp: bool,
) -> Dict[str, float]:
    """返回训练损失字典（包含total_loss及各分项）"""
    model.train()
    loss_meter = AverageMeter()
    coarse_loss_meter = AverageMeter()
    consist_loss_meter = AverageMeter()
    use_hierarchical = isinstance(criterion, HierarchicalLoss)
    # Wrap the dataloader with tqdm for a progress bar
    pbar = tqdm.tqdm(dataloader, desc='Training', leave=False)
    for batch in pbar:
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.split(':')[0], enabled=use_amp):
            outputs = model(images)
            if use_hierarchical:
                # ===== 层次分类损失计算 =====
                coarse_targets = batch['coarse_target'].to(device, non_blocking=True)
                loss_dict = criterion(
                    fine_logits=outputs['logits'],
                    coarse_logits=outputs['coarse_logits'],
                    fine_targets=targets,
                    coarse_targets=coarse_targets,
                    implied_coarse_probs=outputs.get('implied_coarse_probs'),
                )
                fine_loss = loss_dict['fine_loss']
                # 处理per-sample fine_loss (CB-Focal返回per-sample)
                if fine_loss.dim() > 0:
                    if cvar.alpha > 0:
                        fine_loss = cvar(fine_loss)
                    else:
                        fine_loss = fine_loss.mean()
                
                loss = (fine_loss 
                        + loss_dict['lambda_coarse'] * loss_dict['coarse_loss']
                        + loss_dict['lambda_consist'] * loss_dict['consist_loss'])
                
                coarse_loss_meter.update(loss_dict['coarse_loss'].item(), targets.size(0))
                consist_loss_meter.update(loss_dict['consist_loss'].item(), targets.size(0))
            else:
                # ===== 原有损失计算逻辑（向后兼容）=====
                if isinstance(criterion, nn.CrossEntropyLoss):
                    loss = criterion(outputs['logits'], targets)
                else:
                    per_sample_loss = criterion(outputs['logits'], targets)
                    if cvar.alpha <= 0:
                        loss = per_sample_loss.mean()
                    else:
                        loss = cvar(per_sample_loss)
        optimizer.zero_grad()
        if scaler is not None and use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        loss_meter.update(loss.item(), targets.size(0))
        
        # 进度条显示
        postfix = {'loss': f'{loss_meter.avg:.4f}'}
        if use_hierarchical:
            postfix['c_loss'] = f'{coarse_loss_meter.avg:.4f}'
        pbar.set_postfix(postfix)
    
    # 返回损失字典
    result = {'total_loss': loss_meter.avg}
    if use_hierarchical:
        result['coarse_loss'] = coarse_loss_meter.avg
        result['consist_loss'] = consist_loss_meter.avg
    return result


def build_ood_loader(root: Path, data_config: DataConfig, device: str) -> DataLoader:
    transform = build_transforms(data_config.image_size, is_train=False)
    dataset = CitrusDataset(Path(root), None, transform)
    return DataLoader(
        dataset,
        batch_size=data_config.batch_size,
        shuffle=False,
        num_workers=data_config.num_workers,
        pin_memory=device.startswith('cuda'),
    )


def run_tent(model: CitrusHVT, dataloader: DataLoader, device: str, cfg: Dict) -> Dict[str, float]:
    state_backup = copy.deepcopy(model.state_dict())
    for p in model.parameters():
        p.requires_grad_(False)
    params = []
    for module in model.modules():
        if isinstance(module, (nn.BatchNorm2d, nn.LayerNorm)):
            for p in module.parameters():
                p.requires_grad_(True)
                params.append(p)
    if not params:
        raise RuntimeError('No adaptive parameters found for TENT.')
    optimizer = torch.optim.SGD(params, lr=cfg.get('lr', 1e-4))
    preds_all = []
    targets_all = []
    for batch in dataloader:
        images = batch['image'].to(device)
        logits = tent_step(model, images, optimizer)
        preds = logits.argmax(dim=-1).cpu()
        preds_all.append(preds)
        targets_all.append(batch['target'])
    preds_tensor = torch.cat(preds_all)
    targets_tensor = torch.cat(targets_all)
    metrics = {
        'acc': accuracy(preds_tensor, targets_tensor),
        'macro_f1': macro_f1(preds_tensor, targets_tensor, model.config.num_classes),
    }
    model.load_state_dict(state_backup)
    return metrics


def run_training(
    config: Dict,
    data_root: Path,
    train_json: Optional[Path],
    output_dir: Path,
    device: str,
    baseline: Optional[str] = None,
    epochs_override: Optional[int] = None,
    batch_size_override: Optional[int] = None,
    eval_mode: str = 'standard',
    ood_root: Optional[Path] = None,
    resume: Optional[Path] = None,
    seed: int = 42,
) -> Dict[str, Dict]:
    cfg = copy.deepcopy(config)
    seed_everything(seed)
    data_cfg = cfg.get('data', {}).copy()
    data_cfg['root'] = str(data_root)
    if train_json:
        data_cfg['train_json'] = str(train_json)
    if batch_size_override:
        data_cfg['batch_size'] = batch_size_override
    if epochs_override:
        cfg.setdefault('train', {})['epochs'] = epochs_override
    if baseline:
        cfg.setdefault('model', {})['baseline'] = baseline

    if platform.system() == 'Darwin':
        data_cfg['num_workers'] = 0
    data_config = DataConfig(**data_cfg)
    pin_memory = device.startswith('cuda')
    train_loader, val_loader, dataset_targets, coarse_info = build_dataloaders(data_config, seed=seed, pin_memory=pin_memory)
    num_classes = len(train_loader.dataset.classes)
    cfg.setdefault('model', {})['num_classes'] = num_classes
    cfg['model']['img_size'] = cfg.get('image_size', data_config.image_size)


    # 层次分类配置（柑橘病害分类学知识编码）
    use_hierarchical = cfg.get('hierarchical', {}).get('enabled', False)
    if use_hierarchical:
        cfg['model']['use_hierarchical'] = True
        cfg['model']['num_coarse_classes'] = coarse_info['num_coarse_classes']
        cfg['model']['fine_to_coarse'] = coarse_info['fine_to_coarse']
        print(f"✅ 启用层次分类: {coarse_info['num_coarse_classes']}个粗类别 → {num_classes}个细类别")

    
    model = CitrusHVT(ModelConfig(**cfg['model'])).to(device)

    samples_per_class = Counter(dataset_targets)
    ordered_counts = [samples_per_class.get(i, 0) for i in range(num_classes)]
    loss_cfg = cfg.get('loss', {})
    # 支持不同的损失函数类型
    loss_type = loss_cfg.get('loss_type', 'cb_focal')  # 默认用CB-Focal
    
    if loss_type == 'ce':
        # 标准交叉熵
        criterion = nn.CrossEntropyLoss()
    elif loss_type == 'ce_weighted':
        # CE + 类别权重
        class_weights = compute_class_weights(ordered_counts, beta=0.99)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    elif loss_type == 'focal':
        # Focal Loss (无类别平衡)
        criterion = ClassBalancedFocalLoss(
            ordered_counts,
            beta=0.0,  # 不使用类别平衡
            gamma=loss_cfg.get('gamma', 2.0),
            label_smoothing=loss_cfg.get('label_smoothing', 0.0),
        )
    elif loss_type == 'cb_focal':
        # Class-Balanced Focal Loss
        criterion = ClassBalancedFocalLoss(
            ordered_counts,
            beta=loss_cfg.get('class_balanced_beta', 0.99),
            gamma=loss_cfg.get('gamma', 2.0),
            label_smoothing=loss_cfg.get('label_smoothing', 0.0),
        )
    else:
        # 默认CB-Focal
        criterion = ClassBalancedFocalLoss(
            ordered_counts,
            beta=loss_cfg.get('class_balanced_beta', 0.99),
            gamma=loss_cfg.get('gamma', 2.0),
            label_smoothing=loss_cfg.get('label_smoothing', 0.0),
        )
    # CVaR聚合器（只有当使用per-sample loss时才需要）
    cvar_alpha = loss_cfg.get('cvar_alpha', 0.2)
    cvar = CVarAggregator(alpha=cvar_alpha) if not isinstance(criterion, nn.CrossEntropyLoss) else CVarAggregator(alpha=0.0)
    # 如果启用层次分类，用HierarchicalLoss包装
    if use_hierarchical:
        hier_cfg = cfg.get('hierarchical', {})
        criterion = HierarchicalLoss(
            fine_criterion=criterion,
            num_coarse_classes=coarse_info['num_coarse_classes'],
            lambda_coarse=hier_cfg.get('lambda_coarse', 0.3),
            lambda_consist=hier_cfg.get('lambda_consist', 0.1),
        )
        print(f"  层次损失权重: λ_coarse={hier_cfg.get('lambda_coarse', 0.3)}, λ_consist={hier_cfg.get('lambda_consist', 0.1)}")

    train_cfg = cfg.get('train', {})
    optimizer = build_optimizer(
        model,
        lr=train_cfg.get('lr', 3e-4),
        weight_decay=train_cfg.get('weight_decay', 0.05),
    )
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=train_cfg.get('epochs', 80),
        eta_min=cfg.get('scheduler', {}).get('min_lr', 5e-6),
    )
    use_amp = train_cfg.get('use_amp', True) and device.startswith('cuda') and torch.cuda.is_available()
    AmpGradScaler = getattr(torch.amp, 'GradScaler', torch.cuda.amp.GradScaler)
    scaler = AmpGradScaler(enabled=use_amp) if device.startswith('cuda') else None

    start_epoch = 0
    if resume and Path(resume).exists():
        checkpoint = torch.load(resume, map_location='cpu')
        _unwrap_model(model).load_state_dict(checkpoint['model'])
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        start_epoch = checkpoint['epoch'] + 1
        print(f'Resumed from {resume} at epoch {start_epoch}')

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    history: List[Dict[str, float]] = []
    total_epochs = train_cfg.get('epochs', 80)
    # Early stopping和收敛检测参数
    early_stopping_patience = train_cfg.get('early_stopping_patience', 10)
    best_val_acc = 0.0
    best_val_f1 = 0.0
    patience_counter = 0
    plateau_counter = 0
    plateau_threshold = 3  # 连续3个epoch不提升视为平台期
    for epoch in range(start_epoch, total_epochs):
        train_result = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            cvar,
            device=device,
            scaler=scaler,
            use_amp=use_amp,
        )
        # train_one_epoch 现在返回dict
        train_loss = train_result['total_loss'] if isinstance(train_result, dict) else train_result
        
        scheduler.step()
        # 评估模型（包含验证损失）
        val_metrics = evaluate_model(
            model, val_loader, device, num_classes, 
            criterion=criterion, cvar=cvar, compute_val_loss=True
        )
        val_metrics['train_loss'] = train_loss  # 添加训练损失
        # 添加层次分类的训练损失分项
        if isinstance(train_result, dict):
            val_metrics['train_coarse_loss'] = train_result.get('coarse_loss', 0.0)
            val_metrics['train_consist_loss'] = train_result.get('consist_loss', 0.0)
        
        history.append(val_metrics)
        # 获取当前指标
        current_lr = optimizer.param_groups[0]['lr']
        val_acc = val_metrics['acc']
        val_f1 = val_metrics['macro_f1']
        val_loss = val_metrics.get('val_loss', float('inf'))
        
        # 计算训练/验证损失差距（过拟合指标）
        loss_gap = train_loss - val_loss if val_loss != float('inf') else float('inf')
        overfit_ratio = val_loss / train_loss if val_loss != float('inf') and train_loss > 0 else float('inf')
        
        # 判断是否提升
        improved = False
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            improved = True
            patience_counter = 0
            plateau_counter = 0
            # 保存最佳模型
            save_checkpoint(model, optimizer, epoch, output_dir / 'best.pt')
        else:
            patience_counter += 1
            plateau_counter += 1
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
        
        # 打印详细信息
        status_icon = "↑" if improved else "→"
        overfit_warning = ""
        if val_loss != float('inf'):
            if overfit_ratio > 1.5:
                overfit_warning = " ⚠️过拟合"
            elif overfit_ratio < 0.8:
                overfit_warning = " ✓良好"
        
        plateau_warning = ""
        if plateau_counter >= plateau_threshold:
            plateau_warning = f" ⚠️平台期({plateau_counter}epochs)"
        
        lr_warning = ""
        if current_lr < 1e-6:
            lr_warning = " ⚠️LR很小"
         # 格式化验证损失
        val_loss_str = f"{val_loss:.4f}" if val_loss != float('inf') else "N/A"
        # 层次分类额外指标
        hier_info = ""
        if use_hierarchical:
            coarse_acc = val_metrics.get('coarse_acc', 0.0)
            hier_info = f" | Coarse Acc {coarse_acc:.4f}"
            if isinstance(train_result, dict) and 'coarse_loss' in train_result:
                hier_info += f" | CoarseLoss {train_result['coarse_loss']:.4f}"
        
        print(
            f"Epoch {epoch + 1}/{total_epochs} {status_icon} | "
            f"TrainLoss {train_loss:.4f} | ValLoss {val_loss_str} | "
            f"Val Acc {val_acc:.4f} | Val Macro-F1 {val_f1:.4f} | "
            f"Worst-Group F1 {val_metrics['worst_group_f1']:.4f} | "
            f"LR {current_lr:.2e}{hier_info}{lr_warning}"
        )
        # 打印诊断信息
        if val_loss != float('inf'):
            print(f"  📊 损失差距: {loss_gap:.4f} | 过拟合比: {overfit_ratio:.2f}{overfit_warning}")
        if plateau_counter > 0:
            print(f"  📈 最佳Val Acc: {best_val_acc:.4f} | 最佳Val F1: {best_val_f1:.4f}{plateau_warning}")
        
        # Early stopping检查
        if patience_counter >= early_stopping_patience:
            print(f"\n⏹️  Early stopping triggered! No improvement for {early_stopping_patience} epochs.")
            print(f"   Best Val Acc: {best_val_acc:.4f} | Best Val F1: {best_val_f1:.4f}")
            break
        
        # 保存最新checkpoint
        save_checkpoint(model, optimizer, epoch, output_dir / 'last.pt')

    extra_metrics: Dict[str, Dict] = {}
    if eval_mode == 'degradation':
        extra_metrics = evaluate_degradations(model, val_loader, device, num_classes, cfg.get('robust_eval', {}))
        print('Degradation metrics:', extra_metrics)
    elif eval_mode == 'ood':
        if ood_root is None:
            raise ValueError('ood_root is required for OOD evaluation.')
        ood_loader = build_ood_loader(Path(ood_root), data_config, device)
        extra_metrics = evaluate_ood(model, val_loader, ood_loader, device)
        print('OOD metrics:', extra_metrics)
    elif eval_mode == 'tent':
        tent_cfg = cfg.get('tent', {'lr': 1e-4})
        extra_metrics = run_tent(model, val_loader, device, tent_cfg)
        print('TENT metrics:', extra_metrics)

    return {'val': history[-1] if history else {}, 'history': history, 'extra': extra_metrics}

## 12. Default experiment configuration

In [12]:
DEFAULT_CONFIG = {
    'seed': 42,
    'image_size': 224,
    'train': {
        'epochs': 80,
        'batch_size': 32,
        'num_workers': 2, # Reduced based on warning
        'lr': 3.0e-4,
        'weight_decay': 0.05,
        'use_amp': True,
        'early_stopping_patience': 10,  # Early stopping patience
    },
    'scheduler': {
        'min_lr': 5.0e-6,
    },
    'data': {
        'root': str(DATA_ROOT),
        'train_json': str(TRAIN_JSON),
        'val_split': 0.15,
        'batch_size': 32,
        'num_workers': 2, # Reduced based on warning
        'image_size': 224,
    },
    'model': {
        'baseline': 'full',
        'd_model': 384,  # 降低以匹配 ViT-Tiny
        'local_dim': 256,  # 降低以加速训练
        'global_dim': 192,  # 匹配 ViT-Tiny 的输出维度
        'vit_name': 'vit_tiny_patch16_224',  # 切换到 ViT-Tiny
        'drop_path': 0.1,
    },
    'loss': {
        'class_balanced_beta': 0.99,
        'gamma': 2.0,
        'cvar_alpha': 0.2,
        'label_smoothing': 0.0,
    },
    # 层次分类配置（柑橘病害分类学知识编码）
    # 模拟植物病理专家的分层诊断：先判大类(缺素/病害/虫害)，再判细类
    'hierarchical': {
        'enabled': True,           # 是否启用层次分类
        'lambda_coarse': 0.3,      # 粗分类损失权重
        'lambda_consist': 0.1,     # 层次一致性损失权重
    },
    'robust_eval': {
        'jpeg_quality': [20, 40, 60],
        'gaussian_blur_sigma': [0.0, 0.5, 1.0, 2.0],
        'gaussian_noise_std': [0.0, 0.05, 0.1, 0.2],
        'color_jitter_levels': [1, 2, 3],
    },
    'tent': {
        'lr': 1.0e-4,
    },
}

DEFAULT_CONFIG['data']['root'] = str(DATA_ROOT)
DEFAULT_CONFIG['data']['train_json'] = str(TRAIN_JSON)

## 13. Launch training / evaluation

In [13]:
import timm
import copy
import torch
import torch.nn as nn
import pandas as pd
from collections import Counter
from pathlib import Path
import platform
import tqdm # Ensure tqdm is imported for progress bar
from typing import Dict, List, Optional # Ensure these are imported
from torch.optim.lr_scheduler import CosineAnnealingLR # Ensure this is imported


def run_timm_model_training(
    config: Dict,
    model_name: str,
    data_root: Path,
    train_json: Optional[Path],
    output_dir: Path,
    device: str,
    epochs_override: Optional[int] = None,
    batch_size_override: Optional[int] = None,
    seed: int = 42,
) -> Dict[str, Dict]:
    cfg = copy.deepcopy(config)
    seed_everything(seed)

    data_cfg = cfg.get('data', {}).copy()
    data_cfg['root'] = str(data_root)
    if train_json:
        data_cfg['train_json'] = str(train_json)
    if batch_size_override:
        data_cfg['batch_size'] = batch_size_override
    if platform.system() == 'Darwin':
        data_cfg['num_workers'] = 0
    data_config = DataConfig(**data_cfg)

    pin_memory = device.startswith('cuda')
    train_loader, val_loader, dataset_targets, coarse_info = build_dataloaders(data_config, seed=seed, pin_memory=pin_memory)
    num_classes = len(train_loader.dataset.classes)

    # Build the timm model
    print(f"Building timm model: {model_name} with {num_classes} classes.")
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes).to(device)

    # --- Loss Function Setup ---
    samples_per_class = Counter(dataset_targets)
    ordered_counts = [samples_per_class.get(i, 0) for i in range(num_classes)]
    loss_cfg = cfg.get('loss', {})
    loss_type = loss_cfg.get('loss_type', 'cb_focal') # Default to cb_focal

    if loss_type == 'ce':
        criterion = nn.CrossEntropyLoss()
    elif loss_type == 'ce_weighted':
        class_weights = compute_class_weights(ordered_counts, beta=0.99).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    elif loss_type == 'focal':
        criterion = ClassBalancedFocalLoss(
            ordered_counts,
            beta=0.0,
            gamma=loss_cfg.get('gamma', 2.0),
            label_smoothing=loss_cfg.get('label_smoothing', 0.0),
        )
    elif loss_type == 'cb_focal':
        criterion = ClassBalancedFocalLoss(
            ordered_counts,
            beta=loss_cfg.get('class_balanced_beta', 0.99),
            gamma=loss_cfg.get('gamma', 2.0),
            label_smoothing=loss_cfg.get('label_smoothing', 0.0),
        )
    else: # Fallback to CB-Focal
        criterion = ClassBalancedFocalLoss(
            ordered_counts,
            beta=loss_cfg.get('class_balanced_beta', 0.99),
            gamma=loss_cfg.get('gamma', 2.0),
            label_smoothing=loss_cfg.get('label_smoothing', 0.0),
        )

    # CVaR aggregator
    cvar_alpha = loss_cfg.get('cvar_alpha', 0.2)
    cvar = CVarAggregator(alpha=cvar_alpha) if not isinstance(criterion, nn.CrossEntropyLoss) else CVarAggregator(alpha=0.0)

    # --- Training Configuration ---
    train_cfg = cfg.get('train', {})
    optimizer = build_optimizer(
        model,
        lr=train_cfg.get('lr', 3e-4),
        weight_decay=train_cfg.get('weight_decay', 0.05),
    )
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=train_cfg.get('epochs', 80) if not epochs_override else epochs_override,
        eta_min=cfg.get('scheduler', {}).get('min_lr', 5e-6),
    )
    use_amp = train_cfg.get('use_amp', True) and device.startswith('cuda') and torch.cuda.is_available()
    AmpGradScaler = getattr(torch.amp, 'GradScaler', torch.cuda.amp.GradScaler)
    scaler = AmpGradScaler(enabled=use_amp) if device.startswith('cuda') else None

    start_epoch = 0
    total_epochs = train_cfg.get('epochs', 80) if not epochs_override else epochs_override

    current_output_dir = output_dir / model_name.replace('/', '_') # Create a specific subdir for each model
    current_output_dir.mkdir(parents=True, exist_ok=True)

    history: List[Dict[str, float]] = []
    best_val_f1 = 0.0
    patience_counter = 0
    early_stopping_patience = train_cfg.get('early_stopping_patience', 10)

    # --- Simplified train_one_epoch for timm models (no hierarchical, no custom outputs) ---
    def _train_one_epoch_timm(
        model: nn.Module,
        dataloader: DataLoader,
        optimizer: torch.optim.Optimizer,
        criterion,
        cvar: CVarAggregator,
        device: str,
        scaler: Optional[GradScaler],
        use_amp: bool,
    ) -> Dict[str, float]:
        model.train()
        loss_meter = AverageMeter()
        pbar = tqdm.tqdm(dataloader, desc=f'Training {model_name}', leave=False)
        for batch in pbar:
            images = batch['image'].to(device, non_blocking=True)
            targets = batch['target'].to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.split(':')[0], enabled=use_amp):
                outputs = model(images) # timm models usually return logits directly
                if isinstance(criterion, nn.CrossEntropyLoss):
                    loss = criterion(outputs, targets)
                else: # For CB-Focal or other custom losses
                    per_sample_loss = criterion(outputs, targets)
                    if per_sample_loss.dim() > 0: # Check if it's per-sample loss
                        if cvar.alpha > 0:
                            loss = cvar(per_sample_loss)
                        else:
                            loss = per_sample_loss.mean()
                    else: # If criterion already returns a scalar loss
                        loss = per_sample_loss

            optimizer.zero_grad()
            if scaler is not None and use_amp:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            loss_meter.update(loss.item(), targets.size(0))
            pbar.set_postfix({'loss': f'{loss_meter.avg:.4f}'})
        return {'total_loss': loss_meter.avg}

    # --- Training Loop ---
    for epoch in range(start_epoch, total_epochs):
        train_result = _train_one_epoch_timm(
            model,
            train_loader,
            optimizer,
            criterion,
            cvar,
            device=device,
            scaler=scaler,
            use_amp=use_amp,
        )
        train_loss = train_result['total_loss']
        scheduler.step()

        # evaluate_model function expects CitrusHVT model outputs implicitly, specifically for hierarchical features.
        # Since these timm models don't have coarse_logits or implied_coarse_probs, we need to ensure evaluate_model handles this.
        # evaluate_model already checks 'use_hierarchical' attribute of the unwrapped model, so it should be fine.
        # However, the model passed to evaluate_model must expose a 'config' attribute if it's expecting to query it.
        # A simpler way: pass num_classes directly and ensure evaluate_model doesn't try to access hierarchical info.

        # A simple model wrapper could be used if evaluate_model were too strict
        # For now, assuming evaluate_model can handle a plain timm model that just outputs logits.
        # If evaluate_model expects model.config.num_classes, we need to mock it.
        # Let's temporarily add a config to the timm model if it doesn't exist for evaluate_model
        if not hasattr(model, 'config'):
            # Create a minimal config object or mock it
            class MockModelConfig: # Defining here to ensure it's available
                def __init__(self, num_classes_val):
                    self.num_classes = num_classes_val
                    self.use_hierarchical = False # Explicitly disable
            model.config = MockModelConfig(num_classes)

        val_metrics = evaluate_model(
            model, val_loader, device, num_classes,
            criterion=criterion, cvar=cvar, compute_val_loss=True
        )
        val_metrics['train_loss'] = train_loss
        history.append(val_metrics)

        current_lr = optimizer.param_groups[0]['lr']
        val_acc = val_metrics['acc']
        val_f1 = val_metrics['macro_f1']
        val_loss = val_metrics.get('val_loss', float('inf'))

        improved = False
        if val_f1 > best_val_f1: # Use F1 for early stopping for class imbalance
            best_val_f1 = val_f1
            improved = True
            patience_counter = 0
            save_checkpoint(model, optimizer, epoch, current_output_dir / 'best.pt')
        else:
            patience_counter += 1

        print(
            f"Epoch {epoch + 1}/{total_epochs} {'↑' if improved else '→'} | "
            f"TrainLoss {train_loss:.4f} | ValLoss {val_loss:.4f} | "
            f"Val Acc {val_acc:.4f} | Val Macro-F1 {val_f1:.4f} | "
            f"Worst-Group F1 {val_metrics['worst_group_f1']:.4f} | "
            f"LR {current_lr:.2e}"
        )
        save_checkpoint(model, optimizer, epoch, current_output_dir / 'last.pt')

        if patience_counter >= early_stopping_patience:
            print(f"⏹️ Early stopping triggered for {model_name}. No improvement for {early_stopping_patience} epochs.")
            break

    return {'val': history[-1] if history else {}, 'history': history}

In [ ]:
# ### 与现有方法对比：3 种基准模型（ResNet-50、EfficientNet-B0、MobileNetV2）
# 表 3-7 用到的 3 个其他基准在此统一训练与评估，同数据、同划分、平坦 17 类。

# %%
import pandas as pd
from torchvision import models as tv_models
from torchvision.datasets.folder import IMG_EXTENSIONS, default_loader, has_file_allowed_extension

def _build_tv_backbone(name: str, num_classes: int) -> nn.Module:
    """构建 torchvision 骨干 + 17 类分类头，forward 返回 {'logits': logits} 以复用 evaluate_model。"""
    name = name.lower().strip()
    if name == 'resnet50':
        m = tv_models.resnet50(weights=None)
        in_feat = m.fc.in_features
        m.fc = nn.Linear(in_feat, num_classes)
    elif name == 'efficientnet_b0':
        m = tv_models.efficientnet_b0(weights=None)
        in_feat = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_feat, num_classes)
    elif name == 'mobilenet_v2':
        m = tv_models.mobilenet_v2(weights=None)
        in_feat = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_feat, num_classes)
    else:
        raise ValueError(f"Unsupported baseline: {name}. Use resnet50, efficientnet_b0, mobilenet_v2.")
    return m


class _FlatBackboneWrapper(nn.Module):
    """包装 torchvision 模型，接口与 CitrusHVT 一致：forward(x) -> {'logits': logits}。"""
    def __init__(self, backbone: nn.Module):
        super().__init__()
        self.backbone = backbone

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        logits = self.backbone(x)
        return {'logits': logits}


def run_baseline_comparison(
    data_root: Path,
    train_json: Path,
    output_dir: Path,
    device: str,
    epochs: int = 10,
    batch_size: int = 32,
    lr: float = 3e-4,
    seed: int = 42,
) -> pd.DataFrame:
    """
    训练 3 种基准模型（ResNet-50、EfficientNet-B0、MobileNetV2），平坦 17 类、同数据同划分。
    用于填表 3-7 与现有方法对比。返回 DataFrame：method, val_acc, macro_f1 等。
    """
    baselines = ['resnet50', 'efficientnet_b0', 'mobilenet_v2']
    data_cfg = {
        'root': str(data_root),
        'train_json': str(train_json),
        'val_split': 0.15,
        'batch_size': batch_size,
        'num_workers': 2,
        'image_size': 224,
    }
    if platform.system() == 'Darwin':
        data_cfg['num_workers'] = 0
    data_config = DataConfig(**data_cfg)
    pin_memory = device.startswith('cuda')
    train_loader, val_loader, dataset_targets, _ = build_dataloaders(data_config, seed=seed, pin_memory=pin_memory)
    num_classes = len(train_loader.dataset.classes)
    results = []
    criterion = nn.CrossEntropyLoss()
    cvar = CVarAggregator(0.0)
    for name in baselines:
        out_sub = output_dir / f'baseline_{name}'
        out_sub.mkdir(parents=True, exist_ok=True)
        print(f"\n{'='*60}\n基准: {name}\n{'='*60}")
        backbone = _build_tv_backbone(name, num_classes)
        model = _FlatBackboneWrapper(backbone).to(device)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.05)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=5e-6)
        use_amp = device.startswith('cuda')
        scaler = GradScaler() if use_amp else None
        best_f1 = 0.0
        for epoch in range(epochs):
            train_one_epoch(model, train_loader, optimizer, criterion, cvar, device=device, scaler=scaler, use_amp=use_amp)
            scheduler.step()
            val_metrics = evaluate_model(model, val_loader, device, num_classes, criterion=criterion, cvar=cvar, compute_val_loss=False)
            if val_metrics['macro_f1'] > best_f1:
                best_f1 = val_metrics['macro_f1']
                torch.save(model.state_dict(), out_sub / 'best.pt')
            print(f"  Epoch {epoch+1}/{epochs} Val Acc={val_metrics['acc']:.4f} Macro-F1={val_metrics['macro_f1']:.4f}")
        # 用最佳 checkpoint 做最终验证
        model.load_state_dict(torch.load(out_sub / 'best.pt', map_location=device))
        final = evaluate_model(model, val_loader, device, num_classes, criterion=None, cvar=None, compute_val_loss=False)
        display_name = {'resnet50': 'ResNet-50', 'efficientnet_b0': 'EfficientNet-B0', 'mobilenet_v2': 'MobileNetV2'}[name]
        results.append({
            'method': display_name,
            'val_acc': round(final['acc'] * 100, 2),
            'macro_f1': round(final['macro_f1'] * 100, 2),
            'worst_group_f1': round(final.get('worst_group_f1', 0) * 100, 2),
        })
        print(f"  -> {display_name}: Val Acc={final['acc']*100:.2f}% Macro-F1={final['macro_f1']*100:.2f}%")
    df = pd.DataFrame(results)
    df.to_csv(output_dir / 'baseline_comparison_table3_7.csv', index=False)
    return df


# 运行与现有方法对比（表 3-7 的 3 个基准）
# 取消下面注释并运行即可得到 Val Acc / Macro-F1，填入论文表 3-7
baseline_comparison_df = run_baseline_comparison(
     data_root=DATA_ROOT,
     train_json=TRAIN_JSON,
     output_dir=OUTPUT_DIR / 'baseline_comparison',
     device='cuda' if torch.cuda.is_available() else 'cpu',
     epochs=10,
 )
print(baseline_comparison_df.to_string(index=False))

检测到已分好的数据集结构: /kaggle/input/citrus-split/Citrus_Split
  训练集目录: /kaggle/input/citrus-split/Citrus_Split/train
  验证集目录: /kaggle/input/citrus-split/Citrus_Split/val
  找到 8868 个训练样本，17 个类别
  训练集类别分布: {0: 715, 1: 1054, 2: 436, 3: 215, 4: 1048, 5: 357, 6: 226, 7: 119, 8: 165, 9: 257, 10: 385, 11: 892, 12: 990, 13: 529, 14: 493, 15: 599, 16: 388}
  找到 1516 个验证样本
  验证集类别分布: {0: 106, 1: 194, 2: 77, 3: 44, 4: 209, 5: 61, 6: 32, 7: 17, 8: 22, 9: 41, 10: 62, 11: 115, 12: 182, 13: 86, 14: 87, 15: 109, 16: 72}
  检测到测试集目录: /kaggle/input/citrus-split/Citrus_Split/test
  找到 1540 个测试样本
  合并后验证集总样本数: 3056
  合并后验证集类别分布: {0: 213, 1: 389, 2: 156, 3: 90, 4: 419, 5: 124, 6: 65, 7: 36, 8: 46, 9: 83, 10: 125, 11: 231, 12: 365, 13: 174, 14: 175, 15: 220, 16: 145}
  📂 检测到平铺目录结构，使用柑橘病害分类学知识推断层次关系
  🌳 层次分类映射（柑橘病害分类学）:
     粗类别 (4): ['健康', '病害', '缺素', '虫害']
     健康 (1类): ['Healthy']
     病害 (7类): ['Algal_Leaf_Spot', 'Anthracnose ★', 'Brown_Spot', 'Citrus_Canker ★', 'Greasy_Spot', 'Melanose ★', 'Sooty_Mold']
     缺素 

Training:  31%|███▏      | 87/278 [01:26<02:27,  1.30it/s, loss=2.4538]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  Epoch 1/10 Val Acc=0.3599 Macro-F1=0.2841


Training:  32%|███▏      | 88/278 [01:22<02:46,  1.14it/s, loss=1.7466]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  Epoch 2/10 Val Acc=0.5069 Macro-F1=0.4380


...

Training:  21%|██        | 58/278 [00:57<04:10,  1.14s/it, loss=1.3758]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  Epoch 4/10 Val Acc=0.5848 Macro-F1=0.5205


Training:  93%|█████████▎| 258/278 [03:50<00:18,  1.06it/s, loss=1.2313]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  Epoch 5/10 Val Acc=0.6214 Macro-F1=0.5700


Training:  15%|█▍        | 41/278 [00:38<03:53,  1.01it/s, loss=1.1086]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:  20%|█▉        | 55/278 [00:51<03:27,  1.07it/s, loss=1.1188]

## 14.5 层次分类消融实验（Hierarchical Classification Ablation）
### 对比有/无层次分类辅助任务的效果，验证柑橘病害分类学知识编码的价值。

In [ ]:
# %% [markdown]
# ## 14.5 层次分类消融实验（Hierarchical Classification Ablation）
# 
# 对比有/无层次分类辅助任务的效果，验证柑橘病害分类学知识编码的价值。
# 
# 实验设计：
# - **baseline (cnn_only)**: 无层次分类
# - **+ hierarchical**: 加入粗分类辅助头（缺素/病害/虫害）+ 层次一致性约束

# %%

'''from torchvision.datasets.folder import IMG_EXTENSIONS, default_loader, has_file_allowed_extension

print("\n" + "="*80)
print("🌳 层次分类消融实验 (Hierarchical Classification Ablation)")
print("="*80)
print("对比: cnn_only (无层次) vs cnn_only + 层次分类 (编码柑橘病害分类学知识)")
print()

ablation_device = 'cuda' if torch.cuda.is_available() else 'cpu'
HIER_ABLATION_EPOCHS = 10
HIER_ABLATION_OUTPUT = OUTPUT_DIR / 'hier_ablation'
HIER_ABLATION_OUTPUT.mkdir(parents=True, exist_ok=True)

hier_ablation_results = []

# 实验1: 基线 (无层次分类)
print(">>> [1/4] cnn_only 基线 (无层次分类)")
cfg_base = copy.deepcopy(DEFAULT_CONFIG)
cfg_base['hierarchical'] = {'enabled': False}
result_base = run_training(
    cfg_base,
    data_root=DATA_ROOT,
    train_json=TRAIN_JSON,
    output_dir=HIER_ABLATION_OUTPUT / 'no_hier',
    device=ablation_device,
    baseline='cnn_only',
    epochs_override=HIER_ABLATION_EPOCHS,
    eval_mode='standard',
    seed=cfg_base.get('seed', 42),
)
row = {'variant': 'cnn_only (baseline)'}
row.update(result_base.get('val', {}))
hier_ablation_results.append(row)
print(f"   结果: Acc={row.get('acc', 0):.4f}, F1={row.get('macro_f1', 0):.4f}\n")

# 实验2: + 层次分类 (λ_coarse=0.3, λ_consist=0.1)
print(">>> [2/4] cnn_only + 层次分类 (λ_coarse=0.3, λ_consist=0.1)")
cfg_hier = copy.deepcopy(DEFAULT_CONFIG)
cfg_hier['hierarchical'] = {'enabled': True, 'lambda_coarse': 0.3, 'lambda_consist': 0.1}
result_hier = run_training(
    cfg_hier,
    data_root=DATA_ROOT,
    train_json=TRAIN_JSON,
    output_dir=HIER_ABLATION_OUTPUT / 'hier_0.3_0.1',
    device=ablation_device,
    baseline='cnn_only',
    epochs_override=HIER_ABLATION_EPOCHS,
    eval_mode='standard',
    seed=cfg_hier.get('seed', 42),
)
row = {'variant': '+ hierarchical (0.3, 0.1)'}
row.update(result_hier.get('val', {}))
hier_ablation_results.append(row)
print(f"   结果: Acc={row.get('acc', 0):.4f}, F1={row.get('macro_f1', 0):.4f}, Coarse Acc={row.get('coarse_acc', 'N/A')}\n")

# 实验3: 仅粗分类辅助 (无一致性约束)
print(">>> [3/4] cnn_only + 仅粗分类辅助 (λ_coarse=0.3, λ_consist=0.0)")
cfg_coarse_only = copy.deepcopy(DEFAULT_CONFIG)
cfg_coarse_only['hierarchical'] = {'enabled': True, 'lambda_coarse': 0.3, 'lambda_consist': 0.0}
result_coarse = run_training(
    cfg_coarse_only,
    data_root=DATA_ROOT,
    train_json=TRAIN_JSON,
    output_dir=HIER_ABLATION_OUTPUT / 'hier_coarse_only',
    device=ablation_device,
    baseline='cnn_only',
    epochs_override=HIER_ABLATION_EPOCHS,
    eval_mode='standard',
    seed=cfg_coarse_only.get('seed', 42),
)
row = {'variant': '+ coarse only (0.3, 0.0)'}
row.update(result_coarse.get('val', {}))
hier_ablation_results.append(row)
print(f"   结果: Acc={row.get('acc', 0):.4f}, F1={row.get('macro_f1', 0):.4f}, Coarse Acc={row.get('coarse_acc', 'N/A')}\n")

# 实验4: 较强粗分类约束 (λ_coarse=0.5)
print(">>> [4/4] cnn_only + 较强层次约束 (λ_coarse=0.5, λ_consist=0.2)")
cfg_strong = copy.deepcopy(DEFAULT_CONFIG)
cfg_strong['hierarchical'] = {'enabled': True, 'lambda_coarse': 0.5, 'lambda_consist': 0.2}
result_strong = run_training(
    cfg_strong,
    data_root=DATA_ROOT,
    train_json=TRAIN_JSON,
    output_dir=HIER_ABLATION_OUTPUT / 'hier_strong',
    device=ablation_device,
    baseline='cnn_only',
    epochs_override=HIER_ABLATION_EPOCHS,
    eval_mode='standard',
    seed=cfg_strong.get('seed', 42),
)
row = {'variant': '+ strong hier (0.5, 0.2)'}
row.update(result_strong.get('val', {}))
hier_ablation_results.append(row)
print(f"   结果: Acc={row.get('acc', 0):.4f}, F1={row.get('macro_f1', 0):.4f}, Coarse Acc={row.get('coarse_acc', 'N/A')}\n")

# 汇总
hier_df = pd.DataFrame(hier_ablation_results).set_index('variant')
print("\n" + "="*80)
print("🌳 层次分类消融实验结果汇总")
print("="*80)
display(hier_df)

# 计算提升
if len(hier_ablation_results) >= 2:
    base_acc = hier_ablation_results[0].get('acc', 0)
    base_f1 = hier_ablation_results[0].get('macro_f1', 0)
    print("\n相比基线的提升:")
    for r in hier_ablation_results[1:]:
        acc_delta = r.get('acc', 0) - base_acc
        f1_delta = r.get('macro_f1', 0) - base_f1
        print(f"  {r['variant']}: Acc {acc_delta:+.4f}, F1 {f1_delta:+.4f}")
'''

## 14.6 层次分类深度分析（混淆矩阵 + 跨类错误率 + Per-class F1）

In [ ]:
'''import pandas as pd
# 汇总
hier_df = pd.DataFrame(hier_ablation_results).set_index('variant')
print("\n" + "="*80)
print("🌳 层次分类消融实验结果汇总")
print("="*80)
display(hier_df)

# 计算提升
if len(hier_ablation_results) >= 2:
    base_acc = hier_ablation_results[0].get('acc', 0)
    base_f1 = hier_ablation_results[0].get('macro_f1', 0)
    print("\n相比基线的提升:")
    for r in hier_ablation_results[1:]:
        acc_delta = r.get('acc', 0) - base_acc
        f1_delta = r.get('macro_f1', 0) - base_f1
        print(f"  {r['variant']}: Acc {acc_delta:+.4f}, F1 {f1_delta:+.4f}")'''

In [ ]:
'''# %% [markdown]
# ## 14.6 层次分类深度分析（混淆矩阵 + 跨类错误率 + Per-class F1）
# 
# 加载消融实验保存的 best.pt，对验证集做推理，输出：
# 1. 两个模型的混淆矩阵（按粗类别分块）
# 2. 跨类错误率对比（核心指标）
# 3. 各类别 F1 对比
# 4. 重庆重点类别的专项分析

# %%
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score as sklearn_f1

# 设置字体（Kaggle 环境）
# Kaggle 上可用的字体：DejaVu Sans, Arial, sans-serif
# 如果图表中有中文标签，可能需要安装中文字体或使用英文标签
plt.rcParams['font.family'] = ['DejaVu Sans', 'Arial', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

print("\n" + "="*80)
print("📊 层次分类深度分析")
print("="*80)

# --- 1. 加载两个模型的 checkpoint ---
HIER_ANALYSIS_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
HIER_ANALYSIS_OUTPUT = HIER_ABLATION_OUTPUT if 'HIER_ABLATION_OUTPUT' in dir() else OUTPUT_DIR / 'hier_ablation'

baseline_ckpt = HIER_ANALYSIS_OUTPUT / 'no_hier' / 'best.pt'
hier_ckpt = HIER_ANALYSIS_OUTPUT / 'hier_0.3_0.1' / 'best.pt'

# 构建数据加载器（只需要一次）
_analysis_cfg = copy.deepcopy(DEFAULT_CONFIG)
_analysis_data_cfg = _analysis_cfg.get('data', {}).copy()
_analysis_data_cfg['root'] = str(DATA_ROOT)
if TRAIN_JSON is not None:
    _analysis_data_cfg['train_json'] = str(TRAIN_JSON)
if platform.system() == 'Darwin':
    _analysis_data_cfg['num_workers'] = 0
_analysis_dc = DataConfig(**_analysis_data_cfg)
_analysis_pin = HIER_ANALYSIS_DEVICE.startswith('cuda')
_a_train_loader, _a_val_loader, _, _a_coarse_info = build_dataloaders(
    _analysis_dc, seed=_analysis_cfg.get('seed', 42), pin_memory=_analysis_pin
)
_a_class_names = _a_train_loader.dataset.classes
_a_num_classes = len(_a_class_names)
_a_fine_to_coarse = _a_coarse_info['fine_to_coarse']
_a_coarse_classes = _a_coarse_info['coarse_classes']

print(f"类别数: {_a_num_classes}, 粗类别: {_a_coarse_classes}")
print(f"验证集样本数: {len(_a_val_loader.dataset)}")


def _load_and_predict(ckpt_path, use_hierarchical=False):
    """加载checkpoint并收集验证集预测"""
    cfg = copy.deepcopy(DEFAULT_CONFIG)
    cfg['model']['baseline'] = 'cnn_only'
    cfg['model']['num_classes'] = _a_num_classes
    cfg['model']['img_size'] = cfg.get('image_size', 224)
    if use_hierarchical:
        cfg['model']['use_hierarchical'] = True
        cfg['model']['num_coarse_classes'] = _a_coarse_info['num_coarse_classes']
        cfg['model']['fine_to_coarse'] = _a_coarse_info['fine_to_coarse']

    # 粗类别中文到英文映射
    COARSE_CLASS_MAPPING = {
        '缺素': 'Nutrient Deficiency',
        '病害': 'Disease',
        '虫害': 'Pest',
        '健康': 'Healthy'
    }
    # 将粗类别转换为英文
    _a_coarse_classes_en = [COARSE_CLASS_MAPPING.get(c, c) for c in _a_coarse_classes]

    
    model = CitrusHVT(ModelConfig(**cfg['model'])).to(HIER_ANALYSIS_DEVICE)
    # 多 GPU 支持：如果有多张 GPU，自动使用 DataParallel
    if HIER_ANALYSIS_DEVICE.startswith('cuda') and torch.cuda.device_count() > 1:
        print(f"🚀 检测到 {torch.cuda.device_count()} 张 GPU，启用 DataParallel")
        model = nn.DataParallel(model)
    
    state = torch.load(ckpt_path, map_location=HIER_ANALYSIS_DEVICE)
    # 处理 DataParallel 包装的模型状态字典
    model_state = state['model']
    if isinstance(model, nn.DataParallel):
        # 如果当前模型是 DataParallel，但保存的状态没有 module. 前缀，需要适配
        if not any(k.startswith('module.') for k in model_state.keys()):
            # 保存的状态没有 module. 前缀，但当前模型需要，所以直接加载到原始模型
            model.module.load_state_dict(model_state)
        else:
            # 保存的状态有 module. 前缀，直接加载
            model.load_state_dict(model_state)
    else:
        # 如果保存的状态有 module. 前缀，但当前模型不是 DataParallel，需要去掉前缀
        if any(k.startswith('module.') for k in model_state.keys()):
            model_state = {k.replace('module.', ''): v for k, v in model_state.items()}
        model.load_state_dict(model_state)
    model.eval()
    
    preds_all, targets_all = [], []
    with torch.no_grad():
        for batch in _a_val_loader:
            images = batch['image'].to(HIER_ANALYSIS_DEVICE)
            logits = model(images)['logits']
            preds_all.append(logits.argmax(dim=-1).cpu())
            targets_all.append(batch['target'])
    return torch.cat(preds_all).numpy(), torch.cat(targets_all).numpy()


# --- 2. 收集预测 ---
models_data = {}

if baseline_ckpt.exists():
    print(f"\n加载基线模型: {baseline_ckpt}")
    base_preds, base_targets = _load_and_predict(baseline_ckpt, use_hierarchical=False)
    models_data[' baseline (on_hier)'] = {'preds': base_preds, 'targets': base_targets}
    print(f"  预测样本数: {len(base_preds)}, 准确率: {(base_preds == base_targets).mean():.4f}")
else:
    print(f"⚠️ 基线 checkpoint 不存在: {baseline_ckpt}")

if hier_ckpt.exists():
    print(f"\n加载层次分类模型: {hier_ckpt}")
    hier_preds, hier_targets = _load_and_predict(hier_ckpt, use_hierarchical=True)
    models_data['+ hier'] = {'preds': hier_preds, 'targets': hier_targets}
    print(f"  预测样本数: {len(hier_preds)}, 准确率: {(hier_preds == hier_targets).mean():.4f}")
else:
    print(f"⚠️ 层次分类 checkpoint 不存在: {hier_ckpt}")

# %%
# --- 3. 混淆矩阵对比 ---
if len(models_data) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(24, 10))
    
    # 简化类名显示
    short_names = [n.replace('Deficiency_', 'Def_').replace('_Leaf_Spot', '_LS') for n in _a_class_names]
    
    for idx, (name, data) in enumerate(models_data.items()):
        cm = confusion_matrix(data['targets'], data['preds'], labels=list(range(_a_num_classes)))
        # 归一化（按行，即按真实类别）
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)
        
        ax = axes[idx]
        sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=short_names, yticklabels=short_names,
                    ax=ax, vmin=0, vmax=1, cbar_kws={'shrink': 0.8})
        ax.set_title(f'Confusion Matrix: {name}', fontsize=14)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.tick_params(axis='both', labelsize=7, rotation=45)
    
    plt.tight_layout()
    _cm_path = HIER_ANALYSIS_OUTPUT / 'confusion_matrix_comparison.png'
    plt.savefig(_cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"混淆矩阵已保存: {_cm_path}")

elif len(models_data) == 1:
    name, data = list(models_data.items())[0]
    cm = confusion_matrix(data['targets'], data['preds'], labels=list(range(_a_num_classes)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    short_names = [n.replace('Deficiency_', 'Def_').replace('_Leaf_Spot', '_LS') for n in _a_class_names]
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=short_names, yticklabels=short_names,
                ax=ax, vmin=0, vmax=1)
    ax.set_title(f'Confusion Matrix: {name}', fontsize=14)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.tight_layout()
    _cm_path = HIER_ANALYSIS_OUTPUT / 'confusion_matrix.png'
    plt.savefig(_cm_path, dpi=150, bbox_inches='tight')
    plt.show()

# %%
# --- 4. 跨类错误率分析（核心指标）---
print("\n" + "="*80)
print("🔍 跨类错误率分析（Cross-Category Error Rate）")
print("="*80)
print("跨类错误 = 把缺素误判为虫害、病害误判为缺素等，在实际应用中会导致完全错误的处置")
print()

_f2c = np.array(_a_fine_to_coarse)  # fine_to_coarse 映射

error_analysis = {}
for name, data in models_data.items():
    preds, targets = data['preds'], data['targets']
    total = len(targets)
    correct = (preds == targets).sum()
    wrong = total - correct
    
    # 跨类错误 vs 同类错误
    wrong_mask = preds != targets
    wrong_preds = preds[wrong_mask]
    wrong_targets = targets[wrong_mask]
    
    pred_coarse = _f2c[wrong_preds]
    true_coarse = _f2c[wrong_targets]
    
    cross_category = (pred_coarse != true_coarse).sum()
    within_category = (pred_coarse == true_coarse).sum()
    
    error_analysis[name] = {
        'total': total,
        'correct': int(correct),
        'wrong': int(wrong),
        'cross_category_errors': int(cross_category),
        'within_category_errors': int(within_category),
        'accuracy': correct / total,
        'error_rate': wrong / total,
        'cross_category_rate': cross_category / total,  # 占总样本
        'cross_in_errors': cross_category / wrong if wrong > 0 else 0,  # 占错误样本
    }
    
    print(f"📋 {name}:")
    print(f"   总样本: {total}, 正确: {correct}, 错误: {wrong}")
    print(f"   跨类错误: {cross_category} ({cross_category/total*100:.2f}% of total, {cross_category/wrong*100:.1f}% of errors)")
    print(f"   同类错误: {within_category} ({within_category/total*100:.2f}% of total, {within_category/wrong*100:.1f}% of errors)")
    print()

# 对比表格
if len(error_analysis) >= 2:
    print("┌─────────────────────┬──────────┬──────────┬──────────────┬──────────────┐")
    print("│ 模型                │ Acc      │ 错误总数  │ 跨类错误     │ 同类错误     │")
    print("├─────────────────────┼──────────┼──────────┼──────────────┼──────────────┤")
    for name, ea in error_analysis.items():
        print(f"│ {name:<19} │ {ea['accuracy']:.4f}   │ {ea['wrong']:<8} │ {ea['cross_category_errors']:<4} ({ea['cross_category_rate']*100:.2f}%)  │ {ea['within_category_errors']:<4} ({ea['within_category_errors']/ea['total']*100:.2f}%)  │")
    print("└─────────────────────┴──────────┴──────────┴──────────────┴──────────────┘")
    
    names = list(error_analysis.keys())
    if len(names) >= 2:
        base_cross = error_analysis[names[0]]['cross_category_errors']
        hier_cross = error_analysis[names[1]]['cross_category_errors']
        reduction = (base_cross - hier_cross) / base_cross * 100 if base_cross > 0 else 0
        print(f"\n🎯 跨类错误变化: {base_cross} → {hier_cross} (变化 {reduction:+.1f}%)")

# %%
# --- 5. Per-Class F1 对比 ---
print("\n" + "="*80)
print("📊 各类别 F1-Score 对比")
print("="*80)

per_class_data = {}
for name, data in models_data.items():
    f1_per_class = sklearn_f1(data['targets'], data['preds'], 
                               labels=list(range(_a_num_classes)), 
                               average=None, zero_division=0)
    per_class_data[name] = f1_per_class

if per_class_data:
    # 表格输出
    print(f"\n{'类别':<25}", end="")
    for name in per_class_data:
        print(f"│ {name:<18}", end="")
    if len(per_class_data) >= 2:
        print("│ 差异", end="")
    print()
    print("-" * (25 + 20 * len(per_class_data) + (8 if len(per_class_data) >= 2 else 0)))
    
    names_list = list(per_class_data.keys())
    cq_basenames = [c.split('/')[-1] for c in _a_class_names]
    
    for i, cls_name in enumerate(_a_class_names):
        basename = cls_name.split('/')[-1]
        coarse = _a_coarse_classes[_a_fine_to_coarse[i]]
        is_cq = basename in CHONGQING_FOCUS_CLASSES
        marker = " ★" if is_cq else ""
        
        print(f"{cls_name[:22]+marker:<25}", end="")
        for name in names_list:
            print(f"│ {per_class_data[name][i]:.4f}           ", end="")
        if len(names_list) >= 2:
            delta = per_class_data[names_list[1]][i] - per_class_data[names_list[0]][i]
            symbol = "↑" if delta > 0.005 else ("↓" if delta < -0.005 else "→")
            print(f"│ {delta:+.4f} {symbol}", end="")
        print()
    
    # Macro-F1
    print("-" * (25 + 20 * len(per_class_data) + (8 if len(per_class_data) >= 2 else 0)))
    print(f"{'Macro-F1':<25}", end="")
    for name in names_list:
        macro = per_class_data[name].mean()
        print(f"│ {macro:.4f}           ", end="")
    if len(names_list) >= 2:
        delta = per_class_data[names_list[1]].mean() - per_class_data[names_list[0]].mean()
        print(f"│ {delta:+.4f}", end="")
    print()

# %%
# --- 6. 重庆重点类别专项分析 ---
print("\n" + "="*80)
print("🏙️ 重庆重点类别 (Chongqing Focus) 专项分析")
print("="*80)

cq_indices = [i for i, c in enumerate(_a_class_names) if c.split('/')[-1] in CHONGQING_FOCUS_CLASSES]
cq_names = [_a_class_names[i] for i in cq_indices]

if per_class_data and cq_indices:
    print(f"重庆重点类别 ({len(cq_indices)}类): {cq_names}\n")
    
    for name, f1_arr in per_class_data.items():
        cq_f1s = [f1_arr[i] for i in cq_indices]
        cq_macro = np.mean(cq_f1s)
        cq_min = np.min(cq_f1s)
        cq_min_name = _a_class_names[cq_indices[np.argmin(cq_f1s)]]
        print(f"  {name}:")
        print(f"    重庆类别 Macro-F1: {cq_macro:.4f}")
        print(f"    最弱类别: {cq_min_name} (F1={cq_min:.4f})")

# %%
# --- 7. 按粗类别的错误分布可视化 ---
if len(models_data) >= 2 and len(_a_coarse_classes) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for idx, (name, data) in enumerate(models_data.items()):
        preds, targets = data['preds'], data['targets']
        
        # 构建粗类别混淆矩阵
        pred_coarse = _f2c[preds]
        true_coarse = _f2c[targets]
        n_coarse = len(_a_coarse_classes)
        cm_coarse = confusion_matrix(true_coarse, pred_coarse, labels=list(range(n_coarse)))
        cm_coarse_norm = cm_coarse.astype(float) / cm_coarse.sum(axis=1, keepdims=True)
        cm_coarse_norm = np.nan_to_num(cm_coarse_norm)
        
        ax = axes[idx]
        sns.heatmap(cm_coarse_norm, annot=True, fmt='.3f', cmap='Oranges',
                    xticklabels=_a_coarse_classes_en, yticklabels=_a_coarse_classes_en,
                    ax=ax, vmin=0, vmax=1)
        ax.set_title(f'Coarse Confusion: {name}', fontsize=13)
        ax.set_xlabel('Predicted Category')
        ax.set_ylabel('True Category')
    
    plt.tight_layout()
    _coarse_cm_path = HIER_ANALYSIS_OUTPUT / 'coarse_confusion_comparison.png'
    plt.savefig(_coarse_cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"粗分类混淆矩阵已保存: {_coarse_cm_path}")

elif len(models_data) == 1:
    name, data = list(models_data.items())[0]
    pred_coarse = _f2c[data['preds']]
    true_coarse = _f2c[data['targets']]
    n_coarse = len(_a_coarse_classes)
    cm_coarse = confusion_matrix(true_coarse, pred_coarse, labels=list(range(n_coarse)))
    cm_coarse_norm = cm_coarse.astype(float) / cm_coarse.sum(axis=1, keepdims=True)
    cm_coarse_norm = np.nan_to_num(cm_coarse_norm)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm_coarse_norm, annot=True, fmt='.3f', cmap='Oranges',
                xticklabels=_a_coarse_classes_en, yticklabels=_a_coarse_classes_en,
                ax=ax, vmin=0, vmax=1)
    ax.set_title(f'Coarse Confusion: {name}', fontsize=13)
    plt.tight_layout()
    _coarse_cm_path = HIER_ANALYSIS_OUTPUT / 'coarse_confusion.png'
    plt.savefig(_coarse_cm_path, dpi=150, bbox_inches='tight')
    plt.show()

print("\n" + "="*80)
print("✅ 层次分类深度分析完成")
print("="*80)'''

In [ ]:
'''# 粗类别中文到英文映射
COARSE_CLASS_MAPPING = {
    '缺素': 'Nutrient Deficiency',
    '病害': 'Disease',
    '虫害': 'Pest',
    '健康': 'Healthy'
}
# 将粗类别转换为英文
_a_coarse_classes_en = [COARSE_CLASS_MAPPING.get(c, c) for c in _a_coarse_classes]
# %%
# --- 6. 重庆重点类别专项分析 ---
print("\n" + "="*80)
print("🏙️ 重庆重点类别 (Chongqing Focus) 专项分析")
print("="*80)

cq_indices = [i for i, c in enumerate(_a_class_names) if c.split('/')[-1] in CHONGQING_FOCUS_CLASSES]
cq_names = [_a_class_names[i] for i in cq_indices]

if per_class_data and cq_indices:
    print(f"重庆重点类别 ({len(cq_indices)}类): {cq_names}\n")
    
    for name, f1_arr in per_class_data.items():
        cq_f1s = [f1_arr[i] for i in cq_indices]
        cq_macro = np.mean(cq_f1s)
        cq_min = np.min(cq_f1s)
        cq_min_name = _a_class_names[cq_indices[np.argmin(cq_f1s)]]
        print(f"  {name}:")
        print(f"    重庆类别 Macro-F1: {cq_macro:.4f}")
        print(f"    最弱类别: {cq_min_name} (F1={cq_min:.4f})")

# %%
# --- 7. 按粗类别的错误分布可视化 ---
if len(models_data) >= 2 and len(_a_coarse_classes) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for idx, (name, data) in enumerate(models_data.items()):
        preds, targets = data['preds'], data['targets']
        
        # 构建粗类别混淆矩阵
        pred_coarse = _f2c[preds]
        true_coarse = _f2c[targets]
        n_coarse = len(_a_coarse_classes)
        cm_coarse = confusion_matrix(true_coarse, pred_coarse, labels=list(range(n_coarse)))
        cm_coarse_norm = cm_coarse.astype(float) / cm_coarse.sum(axis=1, keepdims=True)
        cm_coarse_norm = np.nan_to_num(cm_coarse_norm)
        
        ax = axes[idx]
        sns.heatmap(cm_coarse_norm, annot=True, fmt='.3f', cmap='Oranges',
                    xticklabels=_a_coarse_classes_en, yticklabels=_a_coarse_classes_en,
                    ax=ax, vmin=0, vmax=1)
        ax.set_title(f'Coarse Confusion: {name}', fontsize=13)
        ax.set_xlabel('Predicted Category')
        ax.set_ylabel('True Category')
    
    plt.tight_layout()
    _coarse_cm_path = HIER_ANALYSIS_OUTPUT / 'coarse_confusion_comparison.png'
    plt.savefig(_coarse_cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"粗分类混淆矩阵已保存: {_coarse_cm_path}")

elif len(models_data) == 1:
    name, data = list(models_data.items())[0]
    pred_coarse = _f2c[data['preds']]
    true_coarse = _f2c[data['targets']]
    n_coarse = len(_a_coarse_classes)
    cm_coarse = confusion_matrix(true_coarse, pred_coarse, labels=list(range(n_coarse)))
    cm_coarse_norm = cm_coarse.astype(float) / cm_coarse.sum(axis=1, keepdims=True)
    cm_coarse_norm = np.nan_to_num(cm_coarse_norm)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm_coarse_norm, annot=True, fmt='.3f', cmap='Oranges',
                xticklabels=_a_coarse_classes_en, yticklabels=_a_coarse_classes_en,
                ax=ax, vmin=0, vmax=1)
    ax.set_title(f'Coarse Confusion: {name}', fontsize=13)
    plt.tight_layout()
    _coarse_cm_path = HIER_ANALYSIS_OUTPUT / 'coarse_confusion.png'
    plt.savefig(_coarse_cm_path, dpi=150, bbox_inches='tight')
    plt.show()

print("\n" + "="*80)
print("✅ 层次分类深度分析完成")
print("="*80)'''

### CNN单独测评，证明CB-Focal + CVaR-DRO的创新价值

In [ ]:
"""
训练策略消融实验
对比不同训练策略对性能的影响，证明CB-Focal + CVaR-DRO的创新价值
"""
import copy
from pathlib import Path
from typing import Dict, List
import pandas as pd

# 从主训练脚本导入必要的函数和类
# 假设这些已经在notebook中定义
# from train_script import run_training, DEFAULT_CONFIG, DATA_ROOT, TRAIN_JSON, OUTPUT_DIR

def run_training_strategy_ablation(
    base_config: Dict,
    data_root: Path,
    train_json: Path,
    output_dir: Path,
    device: str,
    epochs: int = 10,
    baseline: str = 'cnn_only',  # 固定架构，只变训练策略
) -> pd.DataFrame:
    """
    对比不同训练策略的效果
    
    训练策略：
    1. ce: 标准交叉熵
    2. ce_weighted: CE + 类别权重
    3. focal: Focal Loss
    4. cb_focal: Class-Balanced Focal Loss
    5. cb_focal_cvar: CB-Focal + CVaR-DRO (你的创新)
    """
    
    strategies = {
        'ce': {
            'loss': 'ce',
            'cvar_alpha': 0.0,
            'use_class_weight': False,
            'gamma': 0.0,
            'beta': 0.0,
        },
        'ce_weighted': {
            'loss': 'ce',
            'cvar_alpha': 0.0,
            'use_class_weight': True,  # 需要实现
            'gamma': 0.0,
            'beta': 0.0,
        },
        'focal': {
            'loss': 'focal',
            'cvar_alpha': 0.0,
            'use_class_weight': False,
            'gamma': 2.0,
            'beta': 0.0,
        },
        'cb_focal': {
            'loss': 'cb_focal',
            'cvar_alpha': 0.0,
            'use_class_weight': False,
            'gamma': 2.0,
            'beta': 0.99,
        },
        'cb_focal_cvar': {
            'loss': 'cb_focal',
            'cvar_alpha': 0.2,  # CVaR-DRO
            'use_class_weight': False,
            'gamma': 2.0,
            'beta': 0.99,
        },
    }
    
    results = []
    
    for strategy_name, strategy_cfg in strategies.items():
        print(f"\n{'='*60}")
        print(f"训练策略: {strategy_name}")
        print(f"配置: {strategy_cfg}")
        print(f"{'='*60}")
        
        # 复制配置
        cfg = copy.deepcopy(base_config)
        cfg['model']['baseline'] = baseline
        
        # 更新损失配置
        cfg['loss'] = {
            'class_balanced_beta': strategy_cfg['beta'],
            'gamma': strategy_cfg['gamma'],
            'cvar_alpha': strategy_cfg['cvar_alpha'],
            'label_smoothing': 0.0,
        }
        
        # 运行训练
        try:
            metrics = run_training(
                cfg,
                data_root=data_root,
                train_json=train_json,
                output_dir=output_dir / f'strategy_{strategy_name}',
                device=device,
                baseline=baseline,
                epochs_override=epochs,
                eval_mode='standard',
            )
            
            val_metrics = metrics.get('val', {})
            results.append({
                'strategy': strategy_name,
                'description': _get_strategy_description(strategy_name),
                'acc': val_metrics.get('acc', 0),
                'macro_f1': val_metrics.get('macro_f1', 0),
                'worst_group_f1': val_metrics.get('worst_group_f1', 0),
                'min_class_f1': val_metrics.get('min_class_f1', 0),
                'f1_std': val_metrics.get('f1_std', 0),
                'coarse_acc': val_metrics.get('coarse_acc', None),
            })
            
            print(f"✅ {strategy_name}: Acc={val_metrics.get('acc', 0):.4f}, "
                  f"Macro-F1={val_metrics.get('macro_f1', 0):.4f}, "
                  f"Worst-Group F1={val_metrics.get('worst_group_f1', 0):.4f}, "
                  f"Min-Class F1={val_metrics.get('min_class_f1', 0):.4f}, "
                  f"F1_std={val_metrics.get('f1_std', 0):.4f}"))
            
        except Exception as e:
            print(f"❌ {strategy_name} 训练失败: {e}")
            results.append({
                'strategy': strategy_name,
                'description': _get_strategy_description(strategy_name),
                'acc': 0,
                'macro_f1': 0,
                'worst_group_f1': 0,
                'min_class_f1': 0,
                'f1_std': 0,
                'coarse_acc': None,
            })
    
    # 转换为DataFrame
    df = pd.DataFrame(results)
    
    # 计算相对提升（相对于标准CE）
    ce_row = df[df['strategy'] == 'ce']
    if len(ce_row) > 0:
        baseline_acc = ce_row['acc'].values[0]
        baseline_worst_f1 = ce_row['worst_group_f1'].values[0]
        baseline_min_f1 = ce_row['min_class_f1'].values[0]
        df['acc_improvement'] = df['acc'] - baseline_acc
        df['acc_improvement_pct'] = (df['acc_improvement'] / (baseline_acc + 1e-8) * 100).round(2)
        df['worst_group_f1_improvement'] = (df['worst_group_f1'] - baseline_worst_f1).round(4)
        df['min_class_f1_improvement'] = (df['min_class_f1'] - baseline_min_f1).round(4)
    
    # 排序
    df = df.sort_values('acc', ascending=False)
    
    return df


def _get_strategy_description(name: str) -> str:
    descriptions = {
        'ce': '标准交叉熵损失',
        'ce_weighted': 'CE + 类别权重',
        'focal': 'Focal Loss (γ=2.0)',
        'cb_focal': 'Class-Balanced Focal Loss',
        'cb_focal_cvar': 'CB-Focal + CVaR-DRO (α=0.2)',
    }
    return descriptions.get(name, name)


# 使用示例（在notebook中）

# 运行训练策略消融实验
strategy_results = run_training_strategy_ablation(
    base_config=DEFAULT_CONFIG,
    data_root=DATA_ROOT,
    train_json=TRAIN_JSON,
    output_dir=OUTPUT_DIR / 'training_strategy_ablation',
    device='cuda',
    epochs=10,
    baseline='cnn_only',  # 固定架构
)

print("\n训练策略对比结果:")
print(strategy_results.to_string(index=False))

# 保存结果
strategy_results.to_csv(OUTPUT_DIR / 'training_strategy_comparison.csv', index=False)


In [ ]:
import pprint
from torchvision.datasets.folder import IMG_EXTENSIONS, default_loader, has_file_allowed_extension

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

config = copy.deepcopy(DEFAULT_CONFIG)

EPOCHS = 10       # e.g., 10 for quick smoke-test
BATCH_SIZE = None   # e.g., 16 if GPU memory is limited
BASELINE = 'full'   # options: full, global_only, cnn_only, wavecnn_only, concat, no_dwt, no_gate, no_multiscale
EVAL_MODE = 'standard'  # options: standard, degradation, ood, tent
RESUME = None  # set to OUTPUT_DIR / 'last.pt' to continue training

metrics = run_training(
    config,
    data_root=DATA_ROOT,
    train_json=TRAIN_JSON,
    output_dir=OUTPUT_DIR,
    device=device,
    baseline=BASELINE,
    epochs_override=EPOCHS,
    batch_size_override=BATCH_SIZE,
    eval_mode=EVAL_MODE,
    ood_root=OOD_ROOT if EVAL_MODE == 'ood' else None,
    resume=RESUME,
    seed=config.get('seed', 42),
)

print('Final validation metrics:')
pprint.pprint(metrics['val'])
if metrics.get('extra'):
    print('Additional metrics:')
    pprint.pprint(metrics['extra'])

In [ ]:
main_metrics_path = OUTPUT_DIR / 'table_main_metrics.csv'
if main_metrics_path.exists():
    df_main_metrics = pd.read_csv(main_metrics_path)
    print(f"Loaded {main_metrics_path}:")
    display(df_main_metrics)
else:
    print(f"File not found: {main_metrics_path}")

In [ ]:
'''import pandas as pd

# 用你真实的指标填这里
results = [{
    "Model": "CitrusHVT (full, CVaR-DRO)",
    "Epochs": 10,
    "Val Acc": 0.8182,
    "Val Macro-F1": 0.8150,
    "Worst-group F1": 0.0290,
    # 如果你有的话，可以顺手把参数量、FLOPs 也加上
    # "Params (M)": 94.0,
    # "FLOPs @224 (G)": 18.2,
}]

df = pd.DataFrame(results)
df.to_csv("table_main_metrics.csv", index=False)
print("saved to table_main_metrics.csv")
'''

## 14. Ablation experiments

Batch-train several baselines (full/global-only/etc.) with a shorter schedule to quantify each component's impact.

In [ ]:
import pandas as pd
ablation_device = 'cuda' if torch.cuda.is_available() else 'cpu'
ABLATION_BASELINES = ['full', 'global_only', 'cnn_only', 'wavecnn_only', 'concat', 'no_dwt', 'no_gate', 'no_multiscale']
ABLATION_EPOCHS = 5  # short runs for comparison
ABLATION_BATCH_SIZE = 32
ABLATION_OUTPUT_ROOT = OUTPUT_DIR / 'ablations'
ABLATION_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ablation_summary = []
for baseline in ABLATION_BASELINES:
    print(f">>> Training baseline={baseline} for {ABLATION_EPOCHS} epochs")
    cfg = copy.deepcopy(DEFAULT_CONFIG)
    result = run_training(
        cfg,
        data_root=DATA_ROOT,
        train_json=TRAIN_JSON,
        output_dir=ABLATION_OUTPUT_ROOT / baseline,
        device=ablation_device,
        baseline=baseline,
        epochs_override=ABLATION_EPOCHS,
        batch_size_override=ABLATION_BATCH_SIZE,
        eval_mode='standard',
        ood_root=None,
        resume=None,
        seed=cfg.get('seed', 42),
    )
    row = {'baseline': baseline}
    row.update(result.get('val', {}))
    ablation_summary.append(row)

ablation_df = pd.DataFrame(ablation_summary).set_index('baseline')
display(ablation_df)

## 15. Analysis helpers

Utility functions reused by the robustness/diagnostic sections below.

In [ ]:
def load_model_for_analysis(
    base_config: Dict,
    checkpoint_path: Path,
    device: str,
    baseline: Optional[str] = None,
    batch_size_override: Optional[int] = None,
) -> Dict:
    cfg = copy.deepcopy(base_config)
    data_cfg = cfg.get('data', {}).copy()
    data_cfg['root'] = str(DATA_ROOT)
    if TRAIN_JSON is not None:
        data_cfg['train_json'] = str(TRAIN_JSON)
    if batch_size_override:
        data_cfg['batch_size'] = batch_size_override
    if baseline:
        cfg.setdefault('model', {})['baseline'] = baseline
    if platform.system() == 'Darwin':
        data_cfg['num_workers'] = 0
    data_config = DataConfig(**data_cfg)
    pin_memory = device.startswith('cuda')
    train_loader, val_loader, _ = build_dataloaders(data_config, seed=cfg.get('seed', 42), pin_memory=pin_memory)
    class_names = train_loader.dataset.classes
    cfg.setdefault('model', {})['num_classes'] = len(class_names)
    cfg['model']['img_size'] = cfg.get('image_size', data_config.image_size)
    model = CitrusHVT(ModelConfig(**cfg['model'])).to(device)

    ckpt_path = Path(checkpoint_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state['model'])
    model.eval()
    return {
        'model': model,
        'train_loader': train_loader,
        'val_loader': val_loader,
        'data_config': data_config,
        'class_names': class_names,
        'num_classes': len(class_names),
    }


def collect_predictions(model: CitrusHVT, dataloader, device: str) -> Tuple[torch.Tensor, torch.Tensor]:
    preds_all: List[torch.Tensor] = []
    targets_all: List[torch.Tensor] = []
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            logits = model(images)['logits']
            preds_all.append(logits.argmax(dim=-1).cpu())
            targets_all.append(batch['target'])
    return torch.cat(preds_all), torch.cat(targets_all)


def iterate_with_noise(dataloader, noise_std: float):
    for batch in dataloader:
        noisy = batch['image']
        if noise_std > 0:
            noisy = apply_gaussian_noise(noisy, noise_std)
        yield {
            'image': noisy,
            'target': batch['target'],
        }


def tent_noise_sweep(model: CitrusHVT, dataloader, device: str, noise_levels: List[float], cfg: Dict) -> List[Dict]:
    base_state = copy.deepcopy(model.state_dict())
    stats: List[Dict] = []
    for sigma in noise_levels:
        model.load_state_dict(base_state)
        metrics = run_tent(model, iterate_with_noise(dataloader, sigma), device, cfg)
        metrics = dict(metrics)
        metrics['noise_std'] = sigma
        stats.append(metrics)
    model.load_state_dict(base_state)
    model.eval()
    return stats


In [ ]:
"""
在消融实验后分析Gate权重分布
可以直接在消融实验代码块后运行
"""
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import List

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ==================== 配置区域 ====================
# 选择要分析的baseline（必须是包含Gate的模型）
TARGET_BASELINE = 'full'  # 或 'concat', 'no_dwt' 等包含Gate的模型

# Checkpoint路径（消融实验保存的路径）
CHECKPOINT_PATH = ABLATION_OUTPUT_ROOT / TARGET_BASELINE / 'last.pt'

# 类别名称映射（可选，如果不需要可以注释掉）
CHS_TO_ENG = {
    "柑橘缺氮": "Citrus nitrogen deficiency",
    "柑橘缺铁": "Citrus iron deficiency",
    "柑橘缺锌": "Citrus zinc deficiency",
    "柑橘缺锰": "Citrus manganese deficiency",
    "柑橘缺镁": "Citrus magnesium deficiency",
    "柑橘溃疡病": "Citrus canker",
    "柑橘黄龙病": "Huanglongbing (citrus greening)",
    "树脂病": "Citrus gummosis",
    "炭疽病": "Anthracnose",
    "炭疽病+树脂病": "Anthracnose + gummosis",
    "煤烟病": "Sooty mold",
    "脂点黄斑病": "Greasy spot",
    "褐斑病": "Brown spot",
    "褐斑病+树脂病": "Brown spot + gummosis",
    "吹绵蚧": "Cottony cushion scale",
    "柑橘全爪螨": "Citrus red mite",
    "柑橘潜叶蛾": "Citrus leafminer",
    "柑橘锈瘿螨-黑皮果": "Citrus rust mite (black fruit)",
    "橘蚜": "Citrus aphid",
    "红圆蚧": "Red wax scale",
    "黑刺粉虱": "Spiny blackfly",
}
# =================================================

# 检查checkpoint是否存在
if not CHECKPOINT_PATH.exists():
    print(f"❌ Checkpoint not found: {CHECKPOINT_PATH}")
    print(f"   请确保消融实验已经完成，并且baseline='{TARGET_BASELINE}'")
    print(f"   或者修改 TARGET_BASELINE 和 CHECKPOINT_PATH")
else:
    print(f"✅ Loading model from: {CHECKPOINT_PATH}")
    
    # 加载模型和dataloader
    analysis_bundle = load_model_for_analysis(
        DEFAULT_CONFIG,
        checkpoint_path=CHECKPOINT_PATH,
        device=ablation_device,
        baseline=TARGET_BASELINE,
        batch_size_override=ABLATION_BATCH_SIZE,
    )
    
    analysis_model = analysis_bundle['model']
    analysis_val_loader = analysis_bundle['val_loader']
    analysis_device = ablation_device
    analysis_class_names = analysis_bundle['class_names']
    
    print(f"✅ Model loaded successfully")
    print(f"   Baseline: {TARGET_BASELINE}")
    print(f"   Validation batches: {len(analysis_val_loader)}")
    print(f"   Classes: {len(analysis_class_names)}")
    
    # 检查模型是否有Gate
    if not hasattr(analysis_model, 'gate'):
        print(f'\n❌ Current baseline "{TARGET_BASELINE}" does not include the cross-branch gate.')
        print(f'   请选择包含Gate的baseline: full, concat, no_dwt 等')
    else:
        print(f'\n✅ Model has Gate, starting analysis...')
        
        gate_logits: List[torch.Tensor] = []
        targets_for_gate: List[int] = []  # 记录对应的类别
        
        # Hook函数：获取Gate的attention权重logits
        def attn_mlp_hook(module, inputs, output):
            # output is the logits before softmax, shape [B, 2]
            gate_logits.append(output.detach().cpu())
        
        handle = analysis_model.gate.attn_mlp.register_forward_hook(attn_mlp_hook)
        
        # 前向传播，收集Gate权重
        with torch.no_grad():
            for batch in analysis_val_loader:
                analysis_model(batch['image'].to(analysis_device))
                # 记录该batch的标签
                targets_for_gate.extend(batch['target'].tolist())
        
        handle.remove()
        
        if gate_logits:
            logits = torch.cat(gate_logits, dim=0)
            weights = torch.softmax(logits, dim=-1).numpy()  # [N, 2]
            
            # 构造DataFrame
            gate_data = {
                'local_weight': weights[:, 0],
                'global_weight': weights[:, 1],
                'class_idx': targets_for_gate
            }
            
            # 映射类别名称
            try:
                # 尝试使用CHS_TO_ENG映射
                if 'CHS_TO_ENG' in globals():
                    class_names_map = list(CHS_TO_ENG.values())
                    gate_data['class_name'] = [
                        class_names_map[i] if i < len(class_names_map) else f"Class_{i}"
                        for i in targets_for_gate
                    ]
                else:
                    # 使用analysis_class_names
                    gate_data['class_name'] = [
                        analysis_class_names[i] if i < len(analysis_class_names) else f"Class_{i}"
                        for i in targets_for_gate
                    ]
            except Exception as e:
                print(f"Warning: Could not map class names: {e}")
                gate_data['class_name'] = [f"Class_{i}" for i in targets_for_gate]
            
            gate_df = pd.DataFrame(gate_data)
            
            print(f"\n📊 Collected {len(gate_df)} samples")
            print(f"   Local weight range: [{gate_df['local_weight'].min():.3f}, {gate_df['local_weight'].max():.3f}]")
            print(f"   Global weight range: [{gate_df['global_weight'].min():.3f}, {gate_df['global_weight'].max():.3f}]")
            
            # 1. 打印整体统计
            print("\n" + "="*60)
            print("Overall Gate Weight Statistics:")
            print("="*60)
            display(gate_df[['local_weight', 'global_weight']].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
            
            # 2. 绘制整体分布图
            plt.figure(figsize=(8, 5))
            sns.histplot(gate_df['local_weight'], color='tab:blue', label='Local Branch', 
                        kde=True, stat='density', alpha=0.6)
            sns.histplot(gate_df['global_weight'], color='tab:orange', label='Global Branch', 
                        kde=True, stat='density', alpha=0.6)
            plt.legend(fontsize=12)
            plt.title(f'Overall Distribution of Gate Weights ({TARGET_BASELINE})', fontsize=14, fontweight='bold')
            plt.xlabel('Attention Weight', fontsize=12)
            plt.ylabel('Density', fontsize=12)
            plt.grid(alpha=0.3)
            plt.tight_layout()
            output_path = ABLATION_OUTPUT_ROOT / f'gate_weight_distribution_{TARGET_BASELINE}.png'
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"\n✅ Saved: {output_path}")
            plt.show()
            
            # 3. 按类别分布的箱线图
            plt.figure(figsize=(14, 6))
            # 按局部分支权重的中位数排序类别
            sorted_classes = gate_df.groupby('class_name')['local_weight'].median().sort_values(ascending=False).index
            
            sns.boxplot(x='class_name', y='local_weight', data=gate_df, order=sorted_classes, palette="viridis")
            plt.xticks(rotation=45, ha='right', fontsize=10)
            plt.title(f'Local Branch Weight Distribution by Class ({TARGET_BASELINE})', 
                     fontsize=14, fontweight='bold')
            plt.ylabel('Local Branch Weight (0-1)', fontsize=12)
            plt.xlabel('Class', fontsize=12)
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            output_path = ABLATION_OUTPUT_ROOT / f'gate_weight_by_class_{TARGET_BASELINE}.png'
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"✅ Saved: {output_path}")
            plt.show()
            
            # 4. 按类别统计
            print("\n" + "="*60)
            print("Gate Weight Statistics by Class:")
            print("="*60)
            class_stats = gate_df.groupby('class_name').agg({
                'local_weight': ['mean', 'std', 'median'],
                'global_weight': ['mean', 'std', 'median'],
                'class_idx': 'count'
            }).round(4)
            class_stats.columns = ['Local_Mean', 'Local_Std', 'Local_Median', 
                                  'Global_Mean', 'Global_Std', 'Global_Median', 'Count']
            class_stats = class_stats.sort_values('Local_Mean', ascending=False)
            display(class_stats)
            
            # 5. 保存数据
            csv_path = ABLATION_OUTPUT_ROOT / f'gate_weights_{TARGET_BASELINE}.csv'
            gate_df.to_csv(csv_path, index=False)
            print(f"\n✅ Saved gate weights data: {csv_path}")
            
            print("\n" + "="*60)
            print("Interpretation:")
            print("="*60)
            print("• Higher 'Local Branch Weight' means the model relies more on CNN/texture features")
            print("• Higher 'Global Branch Weight' means the model relies more on ViT/global features")
            print("• Values close to 0.5 indicate balanced usage of both branches")
            
        else:
            print('❌ No gate activations captured; ensure the model processes at least one batch.')

## 16. Configure evaluation checkpoint

Load a trained checkpoint once and reuse it across all downstream diagnostics.

In [ ]:
analysis_device = 'cuda' if torch.cuda.is_available() else 'cpu'
ANALYSIS_BASELINE = 'cnn_only'
ANALYSIS_CHECKPOINT = '/kaggle/working/ablations/cnn_only/last.pt'
ANALYSIS_BATCH_SIZE = None  # set to an int to override batch size during analysis

analysis_bundle = load_model_for_analysis(
    DEFAULT_CONFIG,
    checkpoint_path=ANALYSIS_CHECKPOINT,
    device=analysis_device,
    baseline=ANALYSIS_BASELINE,
    batch_size_override=ANALYSIS_BATCH_SIZE,
)
analysis_model = analysis_bundle['model']
analysis_val_loader = analysis_bundle['val_loader']
analysis_class_names = analysis_bundle['class_names']
analysis_num_classes = analysis_bundle['num_classes']
analysis_base_state = copy.deepcopy(analysis_model.state_dict())

print(f"Loaded {ANALYSIS_CHECKPOINT} for baseline={ANALYSIS_BASELINE} on {analysis_device}")
print(f"Validation batches: {len(analysis_val_loader)} | Classes: {analysis_num_classes}")

## 17. JPEG robustness sweep

In [ ]:
JPEG_QUALITIES = [10, 20, 30, 40, 50, 60, 80]  # edit as needed
jpeg_cfg = {'jpeg_quality': JPEG_QUALITIES}

jpeg_metrics = evaluate_degradations(
    analysis_model,
    analysis_val_loader,
    analysis_device,
    analysis_num_classes,
    jpeg_cfg,
)
jpeg_df = pd.DataFrame.from_dict(jpeg_metrics, orient='index')
print('JPEG robustness metrics:')
display(jpeg_df)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display  # 如果在Jupyter环境中使用display函数
plt.figure(figsize=(8, 4))
plt.plot(JPEG_QUALITIES, jpeg_df['macro_f1'], marker='o')
plt.xlabel('JPEG quality')
plt.ylabel('Macro F1')
plt.title('Macro-F1 vs JPEG quality')
plt.grid(True)
plt.savefig('jpeg_robustness.png', dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.show()


## 18. OOD energy diagnostics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns  # 关键：导入seaborn并简写为sns
import pandas as pd
if not OOD_ROOT.exists():
    raise FileNotFoundError(f'OOD dataset not found: {OOD_ROOT}')

ood_loader = build_ood_loader(OOD_ROOT, analysis_bundle['data_config'], analysis_device)
energies_id: List[float] = []
energies_ood: List[float] = []
with torch.no_grad():
    for batch in analysis_val_loader:
        logits = analysis_model(batch['image'].to(analysis_device))['logits']
        energies_id.extend(compute_energy(logits).cpu().tolist())
    for batch in ood_loader:
        logits = analysis_model(batch['image'].to(analysis_device))['logits']
        energies_ood.extend(compute_energy(logits).cpu().tolist())

energy_df = pd.DataFrame(
    {
        'energy': energies_id + energies_ood,
        'split': ['ID'] * len(energies_id) + ['OOD'] * len(energies_ood),
    }
)
plt.figure(figsize=(8, 4))
sns.kdeplot(data=energy_df, x='energy', hue='split', fill=True, common_norm=False)
plt.title('Energy distribution: ID vs OOD')
# 可选：保存图表（延续之前的保存方法，放在plt.show()前）
plt.savefig('energy_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 19. TENT noise robustness

In [ ]:
NOISE_STDS = [0.0, 0.05, 0.1, 0.2]  # Gaussian noise std applied before TENT adaptation
tent_cfg = DEFAULT_CONFIG.get('tent', {'lr': 1e-4})

tent_stats = tent_noise_sweep(analysis_model, analysis_val_loader, analysis_device, NOISE_STDS, tent_cfg)
tent_df = pd.DataFrame(tent_stats).set_index('noise_std')
print('TENT macro metrics vs noise std:')
display(tent_df)
plt.figure(figsize=(8, 4))
plt.plot(tent_df.index, tent_df['macro_f1'], marker='o', label='Macro F1')
plt.plot(tent_df.index, tent_df['acc'], marker='s', label='Accuracy')
plt.xlabel('Noise std')
plt.ylabel('Score')
plt.title('TENT robustness under Gaussian noise')
plt.grid(True)
plt.legend()
plt.savefig('TENT_noise_robustness.png', dpi=300, bbox_inches='tight')
plt.show()

## 20. Gate weight distribution

In [ ]:
if not hasattr(analysis_model, 'gate'):
    print('Current baseline does not include the cross-branch gate.')
else:
    gate_logits: List[torch.Tensor] = []
    def gate_hook(module, inputs, output):
        gate_logits.append(output.detach().cpu())
    handle = analysis_model.gate.attn_mlp.register_forward_hook(gate_hook)
    with torch.no_grad():
        for batch in analysis_val_loader:
            analysis_model(batch['image'].to(analysis_device))
    handle.remove()
    if gate_logits:
        logits = torch.cat(gate_logits, dim=0)
        weights = torch.softmax(logits, dim=-1).numpy()
        gate_df = pd.DataFrame(weights, columns=['local_weight', 'global_weight'])
        display(gate_df.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))
        plt.figure(figsize=(6, 4))
        sns.histplot(gate_df['local_weight'], color='tab:blue', label='Local', kde=True, stat='density')
        sns.histplot(gate_df['global_weight'], color='tab:orange', label='Global', kde=True, stat='density')
        plt.legend()
        plt.title('Distribution of gate attention weights')
        plt.xlabel('Weight')
        plt.savefig('Gate_weight_distribution.png', dpi=300, bbox_inches='tight')
        plt.show()
    else:
        print('No gate activations captured; ensure the model processes at least one batch.')

## 21. Confusion matrix

In [ ]:
# -------------------------- 1. 导入所有必要库 --------------------------
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
from typing import List, Tuple  # 适配函数返回类型注解
from sklearn.metrics import confusion_matrix
import matplotlib.font_manager as fm

# -------------------------- 2. 中文-英文映射表 --------------------------
CHS_TO_ENG = {
    # Nutrient deficiencies
    "柑橘缺氮": "Citrus nitrogen deficiency",
    "柑橘缺铁": "Citrus iron deficiency",
    "柑橘缺锌": "Citrus zinc deficiency",
    "柑橘缺锰": "Citrus manganese deficiency",
    "柑橘缺镁": "Citrus magnesium deficiency",
    # Diseases
    "柑橘溃疡病": "Citrus canker",
    "柑橘黄龙病": "Huanglongbing (citrus greening)",
    "树脂病": "Citrus gummosis",
    "炭疽病": "Anthracnose",
    "炭疽病+树脂病": "Anthracnose + gummosis",
    "煤烟病": "Sooty mold",
    "脂点黄斑病": "Greasy spot",
    "褐斑病": "Brown spot",
    "褐斑病+树脂病": "Brown spot + gummosis",
    # Insect pests
    "吹绵蚧": "Cottony cushion scale",
    "柑橘全爪螨": "Citrus red mite",
    "柑橘潜叶蛾": "Citrus leafminer",
    "柑橘锈瘿螨-黑皮果": "Citrus rust mite (black fruit)",
    "橘蚜": "Citrus aphid",
    "红圆蚧": "Red wax scale",
    "黑刺粉虱": "Spiny blackfly",
}

# -------------------------- 3. 配置参数 --------------------------
analysis_class_names = list(CHS_TO_ENG.values())  # 英文类别名（如需中文，改为 list(CHS_TO_ENG.keys())）
analysis_num_classes = len(CHS_TO_ENG)  # 自动计算17类
preds, targets = collect_predictions(analysis_model, val_loader, analysis_device)
cm = confusion_matrix(targets.numpy(), preds.numpy(), labels=list(range(analysis_num_classes)))
cm_df = pd.DataFrame(cm, index=analysis_class_names, columns=analysis_class_names)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Validation confusion matrix')
plt.tight_layout()
plt.show()
plt.savefig('Confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 统计验证集包含哪些类别
import pandas as pd

# 假设你已经得到了 val_loader（验证集DataLoader）
val_classes = set()  # 用集合存验证集里的类别（去重）
val_class_counts = {}  # 统计每个类别的样本数

for batch in analysis_val_loader:
    targets = batch['target'].numpy()  # 从字典里取真实标签
    for cls in targets:
        val_classes.add(cls)
        val_class_counts[cls] = val_class_counts.get(cls, 0) + 1

# 打印结果
print(f"验证集共包含 {len(val_classes)} 个类别")
print("验证集类别及样本数：")
for cls_idx in sorted(val_class_counts.keys()):
    # 映射到你的类别名（英文/中文均可）
    cls_name = list(CHS_TO_ENG.values())[cls_idx]
    print(f"类别 {cls_idx}：{cls_name} → {val_class_counts[cls_idx]} 个样本")

# 对比所有类别（17类），看哪些没出现在验证集
all_cls_indices = set(range(len(CHS_TO_ENG)))  # 所有类别的索引（0-16）
missing_cls = all_cls_indices - val_classes  # 验证集缺失的类别
if missing_cls:
    print(f"\n验证集缺失的类别（共 {len(missing_cls)} 个）：")
    for cls_idx in sorted(missing_cls):
        cls_name = list(CHS_TO_ENG.values())[cls_idx]
        print(f"类别 {cls_idx}：{cls_name}")